# Entrega 2 — RNN sin memoria (SimpleRNN) y RNN con memoria (LSTM) en 20 Newsgroups

Este notebook ejecuta **todo el plan de la Entrega 2** (`PLAN_ENTREGA2.md`) y al final descarga un archivo
`resultados_e2.zip` con las métricas, tablas, figuras y modelos.

**Cómo usarlo**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución →* **GPU (T4)** → Guardar.
2. *Entorno de ejecución →* **Ejecutar todas**.
3. Esperar a que termine (entre 30 y 60 minutos con GPU; sin GPU puede tardar 2–3 horas). No cierres la pestaña.
4. Al final se descarga `resultados_e2.zip`. **Ponlo en la carpeta `proyecto2`** (junto a `PLAN_ENTREGA2.md`).

Qué hace, en orden: EDA → configuración base de SimpleRNN y LSTM → ajuste de parámetros uno a la vez
(L, unidades, dropout, lr; elegido por F1 macro de validación) → evaluación única en test → semillas 1, 7 y 123 →
MLP Tuned de la E1 re-entrenado en el mismo entorno → tablas, figuras y demo.

Las celdas `%%writefile` contienen exactamente los scripts de `NewsClassifier-main/`.

## 1. Entorno

In [ ]:
!nvidia-smi -L || echo "Sin GPU: funciona, pero será más lento"
import sys, tensorflow as tf, keras, sklearn
print("Python", sys.version.split()[0], "| TensorFlow", tf.__version__, "| Keras", keras.__version__,
      "| scikit-learn", sklearn.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# Opcional: USAR_DRIVE = True guarda todo en Google Drive y permite continuar si Colab se desconecta
# (al volver a ejecutar, las corridas que ya terminaron se saltan).
USAR_DRIVE = False

import os
if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/NewsClassifier_E2"
else:
    WORKDIR = "/content/NewsClassifier_E2"
os.makedirs(WORKDIR + "/results", exist_ok=True)
os.chdir(WORKDIR)
print("Carpeta de trabajo:", os.getcwd())

## 2. Código del proyecto (scripts de `NewsClassifier-main/`)

In [ ]:
%%writefile data_utils.py
"""
================================================================================
PROYECTO PRÁCTICO: REDES NEURONALES - UNIVERSIDAD DEL VALLE (2026-2)
UTILIDADES DE DATOS COMPARTIDAS (ENTREGA 1 Y ENTREGA 2)
================================================================================
Carga 20 Newsgroups exactamente como en la Entrega 1 (mismos documentos, misma
limpieza y misma partición), para que el MLP y las RNN se puedan comparar.
"""

from __future__ import annotations

import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split

# Se quitan encabezados, firmas y citas: scikit-learn lo recomienda para que la
# evaluación sea realista (los metadatos delatan el grupo de origen).
REMOVE = ("headers", "footers", "quotes")
SPLIT_SEED = 42   # Semilla de partición de la E1 (no cambia nunca)
VAL_SIZE = 0.15   # E1: 15 % del train oficial para validación (estratificado)


def load_20newsgroups(data_home: str | None = None) -> dict:
    """
    Devuelve un diccionario con las tres particiones de la E1:
      train: 9 616 documentos | val: 1 698 | test: 7 532 (test oficial, separado por fecha)
    Cada partición es una tupla (lista_de_textos, etiquetas_int32).
    """
    raw_train = fetch_20newsgroups(subset="train", remove=REMOVE, random_state=SPLIT_SEED, data_home=data_home)
    raw_test = fetch_20newsgroups(subset="test", remove=REMOVE, random_state=SPLIT_SEED, data_home=data_home)

    train_texts, val_texts, train_labels, val_labels = train_test_split(
        raw_train.data, raw_train.target,
        test_size=VAL_SIZE, random_state=SPLIT_SEED, stratify=raw_train.target,
    )
    return {
        "train": (list(train_texts), np.asarray(train_labels, dtype=np.int32)),
        "val": (list(val_texts), np.asarray(val_labels, dtype=np.int32)),
        "test": (list(raw_test.data), np.asarray(raw_test.target, dtype=np.int32)),
        "class_names": list(raw_train.target_names),
    }

In [ ]:
%%writefile train_rnn.py
"""
================================================================================
PROYECTO PRÁCTICO: REDES NEURONALES - UNIVERSIDAD DEL VALLE (2026-2)
ENTREGA 2: RNN SIN MEMORIA (SimpleRNN) Y RNN CON MEMORIA (LSTM)
DATASET 3: 20 NEWSGROUPS (20 CATEGORÍAS) | FRAMEWORK: KERAS 3 + TENSORFLOW
================================================================================
Entrena UN modelo recurrente y guarda todo lo necesario para el reporte:

  results/e2/runs/<run_id>/config.json   configuración, versiones y equipo
  results/e2/runs/<run_id>/history.json  métricas por época (para las curvas)
  results/e2/runs/<run_id>/metrics.json  métricas de validación (y de test con --eval-test)
  results/e2/runs/<run_id>/curvas.png    curvas de pérdida y exactitud (épocas desde 1)
  checkpoints/<run_id>.weights.h5        pesos de la mejor época (menor val_loss)
  results/e2/bitacora.csv                una fila por corrida

Ejemplos:
  python train_rnn.py --cell simple --quick-test
  python train_rnn.py --cell lstm --seq-len 200 --units 128 --dropout 0.3
"""

from __future__ import annotations

import argparse
import csv
import json
import os
import platform
import random
import sys
import time
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
# Al cargar pesos en un modelo nuevo, Keras avisa que no restaura el estado del optimizador (no se necesita)
warnings.filterwarnings("ignore", message="Skipping variable loading for optimizer")
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import sklearn
import tensorflow as tf
import keras
from sklearn import metrics as skm

from data_utils import load_20newsgroups


# ==============================================================================
# SECCIÓN 1: CONFIGURACIÓN Y REPRODUCIBILIDAD
# ==============================================================================

# Configuración base (ver PLAN_ENTREGA2.md §4 y §5: de dónde sale cada valor)
DEFAULTS = {
    "cell": "lstm",        # "simple" (RNN sin memoria) o "lstm" (RNN con memoria)
    "seq_len": 200,        # L: tokens por documento (EDA + validación)
    "vocab_size": 20_000,  # V: incluye el relleno ('') y la palabra desconocida ([UNK])
    "embed_dim": 100,      # tamaño del embedding (ejemplo oficial de Keras con 20 Newsgroups)
    "units": 128,          # unidades de la capa recurrente (se ajusta en validación)
    "dropout": 0.3,        # dropout después de la RNN (E1 usó 0.3-0.4; se ajusta en validación)
    "lr": 1e-3,            # Adam, igual que la E1
    "clipnorm": 1.0,       # recorte del gradiente (Pascanu et al., 2013)
    "epochs": 30,          # máximo; EarlyStopping decide cuándo parar
    "batch_size": 64,      # igual que la E1
    "seed": 42,            # igual que la E1
}

CELL_NAMES = {"simple": "SimpleRNN", "lstm": "LSTM"}
COLORS = {"simple": "#2a78d6", "lstm": "#eb6834", "mlp": "#1baf7a"}  # paleta validada (dataviz)
EMPTY_TOKEN = "emptydoc"   # marcador para documentos vacíos (PLAN §3.3)

OUTPUT_ROOT = Path(".")


def set_output_root(root: str | Path) -> None:
    """Permite escribir todo bajo otra carpeta (por ejemplo, para la prueba rápida)."""
    global OUTPUT_ROOT
    OUTPUT_ROOT = Path(root)


def results_dir() -> Path:
    return OUTPUT_ROOT / "results" / "e2"


def runs_dir() -> Path:
    return results_dir() / "runs"


def checkpoints_dir() -> Path:
    return OUTPUT_ROOT / "checkpoints"


def models_dir() -> Path:
    return OUTPUT_ROOT / "models" / "e2"


def set_seed(seed: int) -> None:
    """Fija semillas para que la corrida sea reproducible (mismo criterio que la E1)."""
    random.seed(seed)
    np.random.seed(seed)
    keras.utils.set_random_seed(seed)


def run_id(cfg: dict) -> str:
    """Nombre único y legible de una corrida a partir de su configuración."""
    return (f"{cfg['cell']}_L{cfg['seq_len']}_V{cfg['vocab_size']}_u{cfg['units']}"
            f"_d{cfg['dropout']}_lr{cfg['lr']:g}_s{cfg['seed']}")


def environment_info() -> dict:
    """Versiones y equipo: van al reporte (condiciones de entrenamiento)."""
    cpu = platform.processor() or ""
    try:
        with open("/proc/cpuinfo", encoding="utf-8") as f:
            for line in f:
                if line.startswith("model name"):
                    cpu = line.split(":", 1)[1].strip()
                    break
    except OSError:
        pass
    ram_gb = None
    try:
        import psutil  # disponible en Colab
        ram_gb = round(psutil.virtual_memory().total / 1e9, 1)
    except Exception:
        pass
    gpus = []
    for gpu in tf.config.list_physical_devices("GPU"):
        try:
            gpus.append(tf.config.experimental.get_device_details(gpu).get("device_name", gpu.name))
        except Exception:
            gpus.append(gpu.name)
    return {
        "python": sys.version.split()[0],
        "tensorflow": tf.__version__,
        "keras": keras.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "sistema": f"{platform.system()} {platform.release()}",
        "cpu": cpu,
        "nucleos_cpu": os.cpu_count(),
        "ram_gb": ram_gb,
        "gpu": gpus or None,
    }


# ==============================================================================
# SECCIÓN 2: DE TEXTO A SECUENCIAS
# ==============================================================================

def fill_empty(texts: list[str]) -> list[str]:
    """Documentos vacíos -> token marcador, para que ninguna secuencia quede vacía (PLAN §3.3)."""
    return [t if t.strip() else EMPTY_TOKEN for t in texts]


def to_tensor(texts: list[str]) -> tf.Tensor:
    """Cadenas -> tensor de TensorFlow (evita np.array de texto, que reserva memoria por el texto más largo)."""
    return tf.constant(texts, dtype=tf.string)


class SequenceData:
    """
    Vectorizador ajustado SOLO con train (igual que el TF-IDF de la E1) y las secuencias de
    cada partición. Se guarda la versión de longitud variable y se recorta/rellena a L cuando
    se pide, así varias corridas con distinto L reutilizan el mismo vocabulario.
    """

    def __init__(self, data: dict, vocab_size: int):
        self.vocab_size = vocab_size
        self.labels = {s: data[s][1] for s in ("train", "val", "test")}
        raw = {s: data[s][0] for s in ("train", "val", "test")}
        texts = {s: fill_empty(raw[s]) for s in raw}

        self.vectorizer = keras.layers.TextVectorization(
            max_tokens=vocab_size, output_mode="int", ragged=True)  # minúsculas y sin puntuación (por defecto)
        self.vectorizer.adapt(to_tensor(texts["train"]))
        self.vocab = self.vectorizer.get_vocabulary()
        self.n_tokens = len(self.vocab)   # = vocab_size salvo que train tenga menos palabras distintas
        self.empty_id = self.vocab.index(EMPTY_TOKEN) if EMPTY_TOKEN in self.vocab else 1

        self.ragged = {s: self.vectorizer(to_tensor(texts[s])) for s in texts}
        lengths = {s: self.ragged[s].row_lengths().numpy() for s in texts}
        # Vacíos = texto en blanco + textos que quedan sin tokens al quitar la puntuación
        self.n_empty = {s: int(sum(not t.strip() for t in raw[s]) + int((lengths[s] == 0).sum())) for s in raw}

    def padded(self, split: str, seq_len: int) -> np.ndarray:
        """Matriz (N, L): ceros a la derecha y, si el texto es más largo, los primeros L tokens."""
        X = self.ragged[split].to_tensor(default_value=0, shape=[None, seq_len]).numpy().astype("int32")
        X[(X != 0).sum(axis=1) == 0, 0] = self.empty_id
        return X


# ==============================================================================
# SECCIÓN 3: ARQUITECTURA (SimpleRNN O LSTM, MISMO ESQUELETO)
# ==============================================================================

CELLS = {"simple": keras.layers.SimpleRNN, "lstm": keras.layers.LSTM}


def build_rnn(cell: str, vocab_size: int, emb_dim: int = 100, units: int = 128, dropout: float = 0.3,
              lr: float = 1e-3, clipnorm: float = 1.0, num_classes: int = 20) -> keras.Model:
    """
    Embedding (con máscara para ignorar el relleno) -> capa recurrente -> Dropout -> Softmax.
    Lo único que cambia entre los dos modelos es la capa recurrente: así la diferencia
    entre ellos se debe a la memoria.
    """
    inp = keras.Input(shape=(None,), dtype="int32", name="ids")
    x = keras.layers.Embedding(vocab_size, emb_dim, mask_zero=True, name="embedding")(inp)
    x = CELLS[cell](units, name=cell)(x)              # último estado oculto
    x = keras.layers.Dropout(dropout, name="dropout")(x)
    out = keras.layers.Dense(num_classes, activation="softmax", name="softmax")(x)
    model = keras.Model(inp, out, name=f"{cell}_rnn")
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr, clipnorm=clipnorm),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def count_params(model: keras.Model) -> dict:
    trainable = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    total = int(model.count_params())
    return {"total": total, "entrenables": trainable, "no_entrenables": total - trainable}


# ==============================================================================
# SECCIÓN 4: ENTRENAMIENTO
# ==============================================================================

class EpochLogger(keras.callbacks.Callback):
    """Guarda la duración de cada época y la tasa de aprendizaje usada."""

    def on_train_begin(self, logs=None):
        self.times, self.lrs = [], []

    def on_epoch_begin(self, epoch, logs=None):
        self._t0 = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        self.times.append(time.perf_counter() - self._t0)
        self.lrs.append(float(keras.ops.convert_to_numpy(self.model.optimizer.learning_rate)))


def evaluate(model: keras.Model, X: np.ndarray, y: np.ndarray) -> tuple[dict, np.ndarray]:
    """Mismas métricas que la E1, calculadas a partir de las probabilidades."""
    probs = model.predict(X, batch_size=256, verbose=0)
    y_pred = probs.argmax(axis=1)
    loss = float(-np.mean(np.log(np.clip(probs[np.arange(len(y)), y], 1e-7, 1.0))))
    return {
        "loss": loss,
        "accuracy": float(skm.accuracy_score(y, y_pred)),
        "f1_macro": float(skm.f1_score(y, y_pred, average="macro", zero_division=0)),
        "f1_weighted": float(skm.f1_score(y, y_pred, average="weighted", zero_division=0)),
        "precision_macro": float(skm.precision_score(y, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(skm.recall_score(y, y_pred, average="macro", zero_division=0)),
    }, probs


def plot_curves(history: dict, cell: str, best_epoch: int, path: Path) -> None:
    """Curvas de pérdida y exactitud. Color = modelo; entrenamiento punteado, validación continua."""
    epochs = np.arange(1, len(history["loss"]) + 1)
    color = COLORS[cell]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    panels = [("loss", "val_loss", "Pérdida (entropía cruzada)"), ("accuracy", "val_accuracy", "Exactitud")]
    for ax, (k_tr, k_va, label) in zip(axes, panels):
        ax.plot(epochs, history[k_tr], color=color, lw=2, ls="--", alpha=0.6, label="Entrenamiento")
        ax.plot(epochs, history[k_va], color=color, lw=2, label="Validación")
        ax.axvline(best_epoch, color="#898781", lw=1, ls=":")
        ax.annotate(f"mejor época: {best_epoch}", xy=(best_epoch, 0.98), xycoords=("data", "axes fraction"),
                    xytext=(4, -2), textcoords="offset points", fontsize=8, color="#52514e", va="top")
        ax.set_xlabel("Época")
        ax.set_ylabel(label)
        ax.set_xticks(epochs if len(epochs) <= 15 else np.arange(1, len(epochs) + 1, 2))
        ax.grid(True, color="#e1e0d9", lw=0.6)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            ax.spines[side].set_color("#c3c2b7")
        ax.legend(frameon=False)
    fig.suptitle(f"{CELL_NAMES[cell]}: curvas de pérdida y exactitud", fontsize=11)
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    plt.close(fig)


def append_log(row: dict) -> None:
    """Una fila por corrida en la bitácora (results/e2/bitacora.csv)."""
    path = results_dir() / "bitacora.csv"
    new = not path.exists()
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if new:
            writer.writeheader()
        writer.writerow(row)


def train_and_evaluate(cfg: dict, seq: SequenceData, class_names: list[str], phase: str = "manual",
                       eval_test: bool = False, verbose: int = 2) -> dict:
    """Entrena una configuración, carga los pesos de la mejor época y guarda todo."""
    keras.backend.clear_session()
    set_seed(cfg["seed"])
    rid = run_id(cfg)
    rdir = runs_dir() / rid
    rdir.mkdir(parents=True, exist_ok=True)
    checkpoints_dir().mkdir(parents=True, exist_ok=True)
    ckpt = checkpoints_dir() / f"{rid}.weights.h5"

    print(f"\n=== [{phase}] {rid} ===")
    X_tr, X_va = seq.padded("train", cfg["seq_len"]), seq.padded("val", cfg["seq_len"])
    y_tr, y_va = seq.labels["train"], seq.labels["val"]

    model = build_rnn(cfg["cell"], seq.n_tokens, cfg["embed_dim"], cfg["units"], cfg["dropout"],
                      cfg["lr"], cfg["clipnorm"], num_classes=len(class_names))
    logger = EpochLogger()
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1),
        keras.callbacks.ModelCheckpoint(str(ckpt), monitor="val_loss", save_best_only=True, save_weights_only=True),
        logger,
    ]
    t0 = time.perf_counter()
    hist = model.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=cfg["epochs"],
                     batch_size=cfg["batch_size"], callbacks=callbacks, verbose=verbose)
    train_time = time.perf_counter() - t0
    model.load_weights(ckpt)   # pesos de la época con menor val_loss

    history = {k: [float(v) for v in vals] for k, vals in hist.history.items()}
    history["lr"] = logger.lrs
    history["epoch_time_sec"] = logger.times
    best_epoch = int(np.argmin(history["val_loss"]) + 1)

    val_metrics, _ = evaluate(model, X_va, y_va)
    result = {
        "run_id": rid,
        "phase": phase,
        "config": cfg,
        "params": count_params(model),
        "epochs_run": len(history["loss"]),
        "best_epoch": best_epoch,
        "train_time_sec": train_time,
        "sec_per_epoch": train_time / len(history["loss"]),
        "val": val_metrics,
        "empty_docs": seq.n_empty,
    }
    if eval_test:
        X_te = seq.padded("test", cfg["seq_len"])
        test_metrics, probs = evaluate(model, X_te, seq.labels["test"])
        result["test"] = test_metrics
        np.save(rdir / "test_probs.npy", probs.astype("float32"))

    with open(rdir / "config.json", "w", encoding="utf-8") as f:
        json.dump({"config": cfg, "phase": phase, "entorno": environment_info()}, f, indent=2)
    with open(rdir / "history.json", "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)
    with open(rdir / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)
    plot_curves(history, cfg["cell"], best_epoch, rdir / "curvas.png")

    test = result.get("test", {})
    append_log({
        "run_id": rid, "fecha": datetime.now().strftime("%Y-%m-%d %H:%M"), "fase": phase,
        "celda": CELL_NAMES[cfg["cell"]], "L": cfg["seq_len"], "V": cfg["vocab_size"], "unidades": cfg["units"],
        "dropout": cfg["dropout"], "lr": cfg["lr"], "semilla": cfg["seed"], "epocas": result["epochs_run"],
        "mejor_epoca": best_epoch, "val_loss": round(val_metrics["loss"], 4),
        "val_acc": round(val_metrics["accuracy"], 4), "val_f1_macro": round(val_metrics["f1_macro"], 4),
        "test_acc": round(test["accuracy"], 4) if test else "", "test_f1_macro": round(test["f1_macro"], 4) if test else "",
        "s_por_epoca": round(result["sec_per_epoch"], 2), "parametros": result["params"]["total"],
    })
    print(f"-> val_loss={val_metrics['loss']:.4f} | val_acc={val_metrics['accuracy']:.4f} | "
          f"val_F1={val_metrics['f1_macro']:.4f} | mejor época={best_epoch}/{result['epochs_run']} | "
          f"{result['sec_per_epoch']:.1f} s/época" + (f" | test_acc={test['accuracy']:.4f}" if test else ""))
    return result


# ==============================================================================
# SECCIÓN 5: FUNCIÓN PRINCIPAL (CLI)
# ==============================================================================

def main(argv: list[str] | None = None) -> None:
    p = argparse.ArgumentParser(description="RNN (SimpleRNN o LSTM) para 20 Newsgroups - Entrega 2")
    p.add_argument("--cell", choices=["simple", "lstm"], default=DEFAULTS["cell"])
    p.add_argument("--seq-len", type=int, default=DEFAULTS["seq_len"])
    p.add_argument("--vocab-size", type=int, default=DEFAULTS["vocab_size"])
    p.add_argument("--embed-dim", type=int, default=DEFAULTS["embed_dim"])
    p.add_argument("--units", type=int, default=DEFAULTS["units"])
    p.add_argument("--dropout", type=float, default=DEFAULTS["dropout"])
    p.add_argument("--lr", type=float, default=DEFAULTS["lr"])
    p.add_argument("--clipnorm", type=float, default=DEFAULTS["clipnorm"])
    p.add_argument("--epochs", type=int, default=DEFAULTS["epochs"])
    p.add_argument("--batch-size", type=int, default=DEFAULTS["batch_size"])
    p.add_argument("--seed", type=int, default=DEFAULTS["seed"])
    p.add_argument("--eval-test", action="store_true", help="Evalúa en test (solo para el modelo final)")
    p.add_argument("--output-root", default=".", help="Carpeta donde se escriben results/, models/ y checkpoints/")
    p.add_argument("--quick-test", action="store_true", help="Prueba rápida: 2 épocas, V=5000, L=50")
    a = p.parse_args(argv)

    set_output_root(a.output_root)
    cfg = {"cell": a.cell, "seq_len": a.seq_len, "vocab_size": a.vocab_size, "embed_dim": a.embed_dim,
           "units": a.units, "dropout": a.dropout, "lr": a.lr, "clipnorm": a.clipnorm,
           "epochs": a.epochs, "batch_size": a.batch_size, "seed": a.seed}
    if a.quick_test:
        cfg.update(epochs=2, vocab_size=5000, seq_len=50)

    results_dir().mkdir(parents=True, exist_ok=True)
    data = load_20newsgroups()
    seq = SequenceData(data, cfg["vocab_size"])
    train_and_evaluate(cfg, seq, data["class_names"], phase="manual", eval_test=a.eval_test)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile eda_sequences.py
"""
================================================================================
PROYECTO PRÁCTICO: REDES NEURONALES - UNIVERSIDAD DEL VALLE (2026-2)
ENTREGA 2: EDA MÍNIMA DE SECUENCIAS (justifica L y V)
================================================================================
Mide sobre los mismos datos de la E1:
  - longitud de los documentos en tokens (con la misma estandarización que TextVectorization)
  - % de documentos que se truncarían con cada L candidato
  - cobertura del vocabulario de train sobre validación para cada V candidato
  - documentos vacíos tras quitar encabezados, firmas y citas
Salidas: results/e2/eda.json, results/e2/figures/eda_longitudes_vocabulario.png
         y results/e2/preds/test_lengths.npy (para el análisis por longitud).
"""

from __future__ import annotations

import argparse
import json
import os
from pathlib import Path

import numpy as np

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import keras

from data_utils import load_20newsgroups
from train_rnn import results_dir, set_output_root, to_tensor

L_CANDIDATES = (100, 200, 300, 400)
V_CANDIDATES = (10_000, 20_000, 30_000)


def describe(lengths: np.ndarray) -> dict:
    return {
        "media": float(lengths.mean()),
        "mediana": float(np.median(lengths)),
        "p25": float(np.percentile(lengths, 25)),
        "p75": float(np.percentile(lengths, 75)),
        "p90": float(np.percentile(lengths, 90)),
        "p95": float(np.percentile(lengths, 95)),
        "maximo": int(lengths.max()),
    }


def main(argv: list[str] | None = None, data: dict | None = None) -> dict:
    p = argparse.ArgumentParser(description="EDA de secuencias para la Entrega 2")
    p.add_argument("--output-root", default=".")
    a = p.parse_args(argv)
    set_output_root(a.output_root)
    out = results_dir()
    (out / "figures").mkdir(parents=True, exist_ok=True)
    (out / "preds").mkdir(parents=True, exist_ok=True)

    data = data or load_20newsgroups()
    texts = {s: data[s][0] for s in ("train", "val", "test")}

    # Vectorizador SIN límite de vocabulario, ajustado solo con train (mismo criterio que el modelo)
    vec = keras.layers.TextVectorization(output_mode="int", ragged=True)
    vec.adapt(to_tensor(texts["train"]))
    vocab = vec.get_vocabulary()                       # ordenado por frecuencia: '', [UNK], más frecuente, ...
    ids = {s: vec(to_tensor(texts[s])) for s in texts}
    lengths = {s: ids[s].row_lengths().numpy() for s in texts}

    tr = lengths["train"]
    val_ids = ids["val"].flat_values.numpy()
    coverage = {str(V): float(np.mean((val_ids > 1) & (val_ids < V))) for V in V_CANDIDATES}
    words_mean, words_median = float(tr.mean()), float(np.median(tr))

    eda = {
        "documentos": {s: len(texts[s]) for s in texts},
        "longitud_tokens": {s: describe(lengths[s]) for s in texts},
        "pct_truncados_train": {str(L): float(np.mean(tr > L) * 100) for L in L_CANDIDATES},
        "pct_completos_train": {str(L): float(np.mean(tr <= L) * 100) for L in L_CANDIDATES},
        "vocabulario_train_total": len(vocab) - 2,
        "cobertura_val": coverage,
        "vacios": {s: int((lengths[s] == 0).sum()) for s in texts},
        "razon_S_W": {"S": len(texts["train"]), "W_media": words_mean, "W_mediana": words_median,
                      "S_sobre_W_media": len(texts["train"]) / words_mean,
                      "S_sobre_W_mediana": len(texts["train"]) / words_median},
        "clases": {
            s: {name: int((data[s][1] == i).sum()) for i, name in enumerate(data["class_names"])}
            for s in ("train", "val", "test")
        },
    }
    with open(out / "eda.json", "w", encoding="utf-8") as f:
        json.dump(eda, f, indent=2, ensure_ascii=False)
    np.save(out / "preds" / "test_lengths.npy", lengths["test"].astype("int32"))

    # ---- Figura: (a) histograma de longitudes con los L candidatos; (b) cobertura según V ----
    ink, muted, grid = "#0b0b0b", "#52514e", "#e1e0d9"
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    nz = tr[tr > 0]
    bins = np.logspace(0, np.log10(nz.max() + 1), 50)
    ax1.hist(nz, bins=bins, color="#2a78d6", edgecolor="white", linewidth=0.5)
    ax1.set_xscale("log")
    for k, L in enumerate((100, 200, 400)):
        ax1.axvline(L, color=muted, lw=1, ls="--")
        ax1.annotate(f"L={L}: {eda['pct_completos_train'][str(L)]:.0f} % completos", xy=(L, 0.97 - 0.09 * k),
                     xycoords=("data", "axes fraction"), xytext=(3, 0), textcoords="offset points",
                     fontsize=8, color=muted, va="top")
    ax1.set_xlabel("Tokens por documento (escala log)")
    ax1.set_ylabel("Documentos de entrenamiento")
    ax1.set_title("Longitud de los documentos", fontsize=10, color=ink)

    Vs = np.unique(np.linspace(1_000, min(50_000, len(vocab)), 60).astype(int))
    cov = [np.mean((val_ids > 1) & (val_ids < V)) * 100 for V in Vs]
    ax2.plot(Vs, cov, color="#2a78d6", lw=2)
    for V in V_CANDIDATES:
        if V <= len(vocab):
            c = coverage[str(V)] * 100
            ax2.plot([V], [c], "o", color="#2a78d6", ms=7, mec="white", mew=1.5)
            ax2.annotate(f"{V // 1000} mil: {c:.1f} %", xy=(V, c), xytext=(4, -12),
                         textcoords="offset points", fontsize=8, color=muted)
    ax2.set_xlabel("Tamaño del vocabulario V")
    ax2.set_ylabel("% de palabras de validación cubiertas")
    ax2.set_title("Cobertura del vocabulario", fontsize=10, color=ink)
    for ax in (ax1, ax2):
        ax.grid(True, color=grid, lw=0.6)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            ax.spines[side].set_color("#c3c2b7")
    fig.tight_layout()
    fig.savefig(out / "figures" / "eda_longitudes_vocabulario.png", dpi=200)
    plt.close(fig)

    d = eda["longitud_tokens"]["train"]
    print("\n--- EDA DE SECUENCIAS ---")
    print(f"Documentos: {eda['documentos']}")
    print(f"Longitud en train: media {d['media']:.0f}, mediana {d['mediana']:.0f}, p75 {d['p75']:.0f}, "
          f"p90 {d['p90']:.0f}, máx {d['maximo']}")
    print(f"% sin truncar: {', '.join(f'L={k}: {v:.1f} %' for k, v in eda['pct_completos_train'].items())}")
    print(f"Cobertura en validación: {', '.join(f'V={k}: {v * 100:.1f} %' for k, v in coverage.items())}")
    print(f"Vacíos: {eda['vacios']} | S/W (media) = {eda['razon_S_W']['S_sobre_W_media']:.1f}")
    return eda


if __name__ == "__main__":
    main()

In [ ]:
%%writefile run_e2.py
"""
================================================================================
PROYECTO PRÁCTICO: REDES NEURONALES - UNIVERSIDAD DEL VALLE (2026-2)
ENTREGA 2: EJECUCIÓN COMPLETA (EDA -> AJUSTE -> MODELOS FINALES -> REFERENCIA MLP)
================================================================================
Pasos (PLAN_ENTREGA2.md §3, §6 y §7):
  1. EDA de secuencias (longitudes, vocabulario, vacíos).
  2. Configuración base de SimpleRNN y LSTM (mismos hiperparámetros).
  3. Ajuste de parámetros UNO A LA VEZ (L, unidades, dropout, lr), eligiendo por F1 macro de
     validación. Un cambio se adopta solo si mejora más de 0.5 puntos.
  4. Modelos finales: la mejor configuración de cada uno se evalúa UNA vez en test.
  5. Semillas 1, 7 y 123 de la E1 con la configuración final (media ± desviación).
  6. MLP Tuned de la E1 re-entrenado en el mismo entorno (referencia, análisis y demo).
Si se interrumpe, al volver a ejecutarlo salta todo lo que ya terminó.

Uso:  python run_e2.py                 (ejecución completa)
      python run_e2.py --quick-test    (prueba de humo de pocos minutos, en ./prueba_rapida)
"""

from __future__ import annotations

import argparse
import contextlib
import io
import json
import os
import shutil
import sys
import time
from pathlib import Path

import numpy as np

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import keras

import eda_sequences
import train_rnn as tr
from data_utils import load_20newsgroups

CELLS = ["simple", "lstm"]
BASE = {k: v for k, v in tr.DEFAULTS.items() if k != "cell"}
SEARCH = [                       # orden del PLAN §6
    ("seq_len", [100, 200, 400]),
    ("units", [64, 128, 256]),
    ("dropout", [0.2, 0.3, 0.5]),
    ("lr", [1e-3, 5e-4]),
]
# Por debajo de 0.5 puntos de F1 macro la diferencia se trata como ruido: en la E1 la exactitud
# varió entre ±0.3 y ±0.9 puntos solo por cambiar la semilla.
MIN_GAIN = 0.005
EXTRA_SEEDS = [1, 7, 123]        # mismas semillas de la E1 (42 ya es la principal)
MLP_TUNED_ARGS = ["--vocab-size", "10000", "--hidden-dim", "512", "--dropout", "0.3",
                  "--l2-reg", "0.00005", "--lr", "0.0005", "--epochs", "25"]   # E1, README paso 3


class Tee(io.TextIOBase):
    """Escribe en pantalla y en results/e2/log_ejecucion.txt al mismo tiempo."""

    def __init__(self, *streams):
        self.streams = streams

    def write(self, s):
        for st in self.streams:
            st.write(s)
            st.flush()
        return len(s)

    def flush(self):
        for st in self.streams:
            st.flush()


@contextlib.contextmanager
def working_dir(path: Path):
    prev = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(prev)


def run_or_load(cfg: dict, seq: tr.SequenceData, class_names: list[str], phase: str,
                eval_test: bool = False) -> dict:
    """Entrena la configuración o, si ya existe (mismo run_id), reutiliza su resultado."""
    rid = tr.run_id(cfg)
    mfile = tr.runs_dir() / rid / "metrics.json"
    ckpt = tr.checkpoints_dir() / f"{rid}.weights.h5"
    if mfile.exists() and ckpt.exists():
        with open(mfile, encoding="utf-8") as f:
            res = json.load(f)
        if not eval_test or "test" in res:
            print(f"(ya existe) {rid}: val_F1={res['val']['f1_macro']:.4f}")
            return res
    return tr.train_and_evaluate(cfg, seq, class_names, phase=phase, eval_test=eval_test)


def tune(cell: str, seq: tr.SequenceData, class_names: list[str], base: dict, search: list) -> dict:
    """Ajuste uno a la vez: cada valor final queda con su propia evidencia (F1 macro de validación)."""
    out = tr.results_dir() / f"ajuste_{cell}.json"
    current = dict(base, cell=cell)
    base_res = run_or_load(current, seq, class_names, phase="base")
    log = {"cell": cell, "min_gain": MIN_GAIN, "base_config": dict(current),
           "base_run": base_res["run_id"], "base_val": base_res["val"], "pasos": []}

    for param, values in search:
        trials = []
        for v in values:
            cfg = dict(current, **{param: v})
            res = run_or_load(cfg, seq, class_names, phase=f"ajuste_{param}")
            trials.append({"valor": v, "run_id": res["run_id"], "val_f1_macro": res["val"]["f1_macro"],
                           "val_accuracy": res["val"]["accuracy"], "val_loss": res["val"]["loss"],
                           "mejor_epoca": res["best_epoch"], "s_por_epoca": res["sec_per_epoch"]})
        score = {t["valor"]: t["val_f1_macro"] for t in trials}
        previous = current[param]
        best = max(score, key=score.get)
        gain = score[best] - score[previous]
        adopted = best != previous and gain > MIN_GAIN
        if adopted:
            current[param] = best
        reason = (f"{best} mejora el F1 macro de validación en {gain * 100:.2f} puntos" if adopted else
                  f"ningún valor mejora más de {MIN_GAIN * 100:.1f} puntos; se mantiene {previous}")
        log["pasos"].append({"parametro": param, "pruebas": trials, "anterior": previous,
                             "elegido": current[param], "ganancia": gain, "adoptado": adopted, "razon": reason})
        print(f"[{tr.CELL_NAMES[cell]}] {param}: {reason}")

    final_res = run_or_load(current, seq, class_names, phase="final")
    log["config_final"] = current
    log["final_run"] = final_res["run_id"]
    log["final_val"] = final_res["val"]
    with open(out, "w", encoding="utf-8") as f:
        json.dump(log, f, indent=2)
    return current


def finalize(cell: str, cfg: dict, seq: tr.SequenceData, class_names: list[str]) -> dict:
    """Evalúa UNA vez en test el modelo elegido (los mismos pesos que se vieron en validación) y lo guarda."""
    final_dir = tr.results_dir() / "final"
    final_dir.mkdir(parents=True, exist_ok=True)
    out = final_dir / f"{cell}.json"
    if out.exists():
        with open(out, encoding="utf-8") as f:
            return json.load(f)

    rid = tr.run_id(cfg)
    with open(tr.runs_dir() / rid / "metrics.json", encoding="utf-8") as f:
        run = json.load(f)
    keras.backend.clear_session()
    model = tr.build_rnn(cell, seq.n_tokens, cfg["embed_dim"], cfg["units"], cfg["dropout"],
                         cfg["lr"], cfg["clipnorm"], num_classes=len(class_names))
    model.load_weights(tr.checkpoints_dir() / f"{rid}.weights.h5")

    X_te, y_te = seq.padded("test", cfg["seq_len"]), seq.labels["test"]
    test_metrics, probs = tr.evaluate(model, X_te, y_te)
    y_pred = probs.argmax(axis=1)
    from sklearn import metrics as skm
    per_class = skm.classification_report(y_te, y_pred, target_names=class_names, output_dict=True, zero_division=0)
    cm = skm.confusion_matrix(y_te, y_pred, labels=list(range(len(class_names))))

    preds = tr.results_dir() / "preds"
    preds.mkdir(parents=True, exist_ok=True)
    np.save(preds / f"{cell}_test_probs.npy", probs.astype("float32"))
    np.save(preds / "y_test.npy", y_te.astype("int32"))
    np.save(preds / f"{cell}_confusion.npy", cm)

    # Artefactos para la demo (sin internet): pesos, configuración, vocabulario y clases.
    # Se guardan desde el modelo recién cargado: solo pesos de la red, sin el estado del optimizador.
    md = tr.models_dir()
    md.mkdir(parents=True, exist_ok=True)
    model.save_weights(md / f"{cell}_final.weights.h5")
    with open(md / f"{cell}_final_config.json", "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2)
    vocab_file = md / f"vocab_V{cfg['vocab_size']}.txt"
    if not vocab_file.exists():
        with open(vocab_file, "w", encoding="utf-8") as f:
            f.write("\n".join(seq.vocab))
    with open(md / "class_names.json", "w", encoding="utf-8") as f:
        json.dump(class_names, f, indent=2)

    figs = tr.results_dir() / "figures"
    figs.mkdir(parents=True, exist_ok=True)
    shutil.copy(tr.runs_dir() / rid / "curvas.png", figs / f"curvas_{cell}.png")

    result = {"cell": cell, "run_id": rid, "config": cfg, "params": run["params"],
              "epochs_run": run["epochs_run"], "best_epoch": run["best_epoch"],
              "train_time_sec": run["train_time_sec"], "sec_per_epoch": run["sec_per_epoch"],
              "val": run["val"], "test": test_metrics, "por_clase": per_class,
              "empty_docs": run.get("empty_docs")}
    with open(out, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)
    print(f"\n*** {tr.CELL_NAMES[cell]} FINAL (test, evaluado una vez): exactitud={test_metrics['accuracy']:.4f} | "
          f"F1 macro={test_metrics['f1_macro']:.4f}")
    return result


def extra_seeds(cell: str, cfg: dict, seq: tr.SequenceData, class_names: list[str], seeds: list[int]) -> None:
    """La configuración final con otras semillas, para reportar media ± desviación (como la E1)."""
    with open(tr.results_dir() / "final" / f"{cell}.json", encoding="utf-8") as f:
        main_res = json.load(f)
    rows = [{"seed": cfg["seed"], **{k: main_res["test"][k] for k in ("accuracy", "f1_macro")}}]
    for s in seeds:
        res = run_or_load(dict(cfg, seed=s), seq, class_names, phase="semillas", eval_test=True)
        rows.append({"seed": s, **{k: res["test"][k] for k in ("accuracy", "f1_macro")}})
    summary = {"cell": cell, "por_semilla": rows}
    for k in ("accuracy", "f1_macro"):
        vals = np.array([r[k] for r in rows])
        summary[k] = {"media": float(vals.mean()), "desv": float(vals.std(ddof=1)) if len(vals) > 1 else 0.0}
    with open(tr.results_dir() / "final" / f"{cell}_semillas.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)


def mlp_reference(seeds: list[int], quick: bool) -> None:
    """Re-entrena el MLP Tuned de la E1 con el mismo script (corregido) y en el mismo entorno."""
    import train_mlp
    root = tr.OUTPUT_ROOT.resolve()
    with working_dir(root):
        rows = []
        for s in [42] + seeds:
            name = "tuned_mlp_e2" if s == 42 else f"tuned_mlp_e2_s{s}"
            jfile = Path("results") / f"{name}.json"
            if not jfile.exists():
                print(f"\n=== [mlp] {name} ===")
                args = (["--quick-test"] if quick else MLP_TUNED_ARGS) + ["--output-name", name, "--seed", str(s),
                                                                          "--no-demo"]
                if s == 42:
                    args += ["--probs-out", "results/e2/preds/mlp_test_probs.npy"]
                with contextlib.redirect_stdout(io.StringIO()) if quick else contextlib.nullcontext():
                    train_mlp.main(args)
                keras.backend.clear_session()
            with open(jfile, encoding="utf-8") as f:
                r = json.load(f)
            rows.append({"seed": s, "accuracy": r["accuracy"], "f1_macro": r["f1_macro"]})

        # Pesos y vectorizador para la demo (archivo de pesos: mucho más liviano que el .keras con optimizador)
        md = Path("models") / "e2"
        md.mkdir(parents=True, exist_ok=True)
        if not (md / "mlp_tuned.weights.h5").exists():
            mlp = keras.models.load_model("models/tuned_mlp_e2.keras")
            cfg = {"vocab_size": int(mlp.input_shape[-1]), "hidden_dim": int(mlp.layers[0].units),
                   "dropout": float(mlp.layers[1].rate)}
            with contextlib.redirect_stdout(io.StringIO()):
                fresh = train_mlp.build_mlp_model(cfg["vocab_size"], mlp.output_shape[-1], cfg["hidden_dim"],
                                                  cfg["dropout"])
            fresh.set_weights(mlp.get_weights())          # solo pesos de la red, sin estado del optimizador
            fresh.save_weights(md / "mlp_tuned.weights.h5")
            shutil.copy("models/vectorizer_tuned_mlp_e2.pkl", md / "mlp_vectorizer.pkl")
            with open(md / "mlp_config.json", "w", encoding="utf-8") as f:
                json.dump(cfg, f, indent=2)
        summary = {"por_semilla": rows}
        for k in ("accuracy", "f1_macro"):
            vals = np.array([r[k] for r in rows])
            summary[k] = {"media": float(vals.mean()), "desv": float(vals.std(ddof=1)) if len(vals) > 1 else 0.0}
        with open(Path("results") / "e2" / "final" / "mlp_semillas.json", "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2)


def main(argv: list[str] | None = None) -> None:
    p = argparse.ArgumentParser(description="Entrega 2 completa: EDA, ajuste, modelos finales y referencia MLP")
    p.add_argument("--quick-test", action="store_true", help="Prueba de humo (pocos minutos) en ./prueba_rapida")
    p.add_argument("--output-root", default=None)
    p.add_argument("--skip-seeds", action="store_true", help="No entrena las semillas extra")
    p.add_argument("--skip-mlp", action="store_true", help="No re-entrena el MLP de la E1")
    a = p.parse_args(argv)

    quick = a.quick_test
    root = Path(a.output_root or ("prueba_rapida" if quick else "."))
    root.mkdir(parents=True, exist_ok=True)
    tr.set_output_root(root)
    tr.results_dir().mkdir(parents=True, exist_ok=True)
    (tr.results_dir() / "final").mkdir(parents=True, exist_ok=True)

    base, search, seeds = dict(BASE), SEARCH, EXTRA_SEEDS
    if quick:
        base.update(epochs=2, vocab_size=5000, seq_len=50, units=16)
        search = [("seq_len", [50, 100]), ("units", [16, 32]), ("dropout", [0.3]), ("lr", [1e-3])]
        seeds = [1]

    log_file = open(tr.results_dir() / "log_ejecucion.txt", "a", encoding="utf-8")
    sys.stdout = Tee(sys.__stdout__, log_file)
    t_start = time.perf_counter()
    print(f"\n######## ENTREGA 2 - inicio ({'prueba rápida' if quick else 'ejecución completa'}) ########")
    print(json.dumps(tr.environment_info(), indent=2, ensure_ascii=False))

    data = load_20newsgroups()
    class_names = data["class_names"]
    print(f"Datos: train {len(data['train'][0])} | val {len(data['val'][0])} | test {len(data['test'][0])}")

    # 1. EDA
    eda_sequences.main(["--output-root", str(root)], data=data)

    # 2-4. Base, ajuste y modelos finales
    seq_cache: dict[int, tr.SequenceData] = {}

    def seq_for(cfg):
        V = cfg["vocab_size"]
        if V not in seq_cache:
            seq_cache[V] = tr.SequenceData(data, V)
        return seq_cache[V]

    seq = seq_for(base)
    finals = {}
    for cell in CELLS:
        print(f"\n######## {tr.CELL_NAMES[cell]}: base y ajuste de parámetros ########")
        finals[cell] = tune(cell, seq, class_names, base, search)
    for cell in CELLS:
        finalize(cell, finals[cell], seq, class_names)

    # 5. Semillas extra
    if not a.skip_seeds:
        for cell in CELLS:
            print(f"\n######## {tr.CELL_NAMES[cell]}: semillas {seeds} ########")
            extra_seeds(cell, finals[cell], seq, class_names, seeds)

    # 6. MLP de referencia
    if not a.skip_mlp:
        print("\n######## MLP Tuned (E1) en el mismo entorno ########")
        mlp_reference([] if a.skip_seeds else seeds, quick)

    minutes = (time.perf_counter() - t_start) / 60
    print(f"\n######## ENTREGA 2 - fin ({minutes:.1f} min) ########")
    for cell in CELLS:
        with open(tr.results_dir() / "final" / f"{cell}.json", encoding="utf-8") as f:
            r = json.load(f)
        print(f"{tr.CELL_NAMES[cell]:>9}: L={r['config']['seq_len']} u={r['config']['units']} "
              f"d={r['config']['dropout']} lr={r['config']['lr']:g} -> test exactitud {r['test']['accuracy']:.4f}, "
              f"F1 macro {r['test']['f1_macro']:.4f}")
    sys.stdout = sys.__stdout__
    log_file.close()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile analyze_e2.py
"""
================================================================================
PROYECTO PRÁCTICO: REDES NEURONALES - UNIVERSIDAD DEL VALLE (2026-2)
ENTREGA 2: TABLAS Y FIGURAS DEL REPORTE (a partir de results/)
================================================================================
Lee los resultados que deja run_e2.py y genera, sin re-entrenar nada:
  results/e2/tables/*.tex|.md|.csv   tablas que el reporte incluye con \\input
  results/e2/figures/*.png           figuras del reporte y de la presentación
  results/e2/resumen.json            cifras clave para redactar el análisis
Así cada número del reporte sale de un archivo generado por el código.

Uso:  python analyze_e2.py [--output-root .]
"""

from __future__ import annotations

import argparse
import csv
import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from sklearn import metrics as skm

NAMES = {"mlp": "MLP", "simple": "SimpleRNN", "lstm": "LSTM"}
MLP_E2 = "MLP Tuned (re-entrenado)"   # el MLP de la E1 re-entrenado en el entorno de la E2
COLORS = {"simple": "#2a78d6", "lstm": "#eb6834", "mlp": "#1baf7a"}   # paleta validada (dataviz)
INK, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#e1e0d9", "#c3c2b7"
BLUES = LinearSegmentedColormap.from_list(
    "azul", ["#ffffff", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
PARAM_LABELS = {"seq_len": "Longitud L", "units": "Unidades", "dropout": "Dropout", "lr": "Tasa de aprendizaje"}


# ------------------------------------------------------------------------------
# Utilidades
# ------------------------------------------------------------------------------

def load(path: Path):
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def pct(x: float | None, nd: int = 2) -> str:
    return "—" if x is None else f"{x * 100:.{nd}f}"


def num(n: int | None) -> str:
    return "—" if n is None else f"{n:,}".replace(",", "\\,")


def style(ax) -> None:
    ax.grid(True, color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(AXIS)


def write_table(out: Path, name: str, header: list[str], rows: list[list[str]], align: str,
                bold_rows: set[int] | None = None) -> None:
    """Escribe la misma tabla en LaTeX (booktabs), Markdown y CSV."""
    bold_rows = bold_rows or set()
    tex = [f"\\begin{{tabular}}{{{align}}}", "\\toprule", " & ".join(header) + " \\\\", "\\midrule"]
    for i, r in enumerate(rows):
        cells = [f"\\textbf{{{c}}}" if i in bold_rows and c not in ("", "—") else c for c in r]
        tex.append(" & ".join(cells) + " \\\\")
    tex += ["\\bottomrule", "\\end{tabular}"]
    (out / f"{name}.tex").write_text("\n".join(tex) + "\n", encoding="utf-8")

    def plain(s: str) -> str:
        return (s.replace("\\,", " ").replace("$\\pm$", "±").replace("$\\checkmark$", "✓")
                .replace("\\%", "%").replace("\\_", "_").replace("$\\le$", "≤").replace("$>$", ">"))

    md = ["| " + " | ".join(plain(h) for h in header) + " |", "|" + "---|" * len(header)]
    md += ["| " + " | ".join(plain(c) for c in r) + " |" for r in rows]
    (out / f"{name}.md").write_text("\n".join(md) + "\n", encoding="utf-8")
    with open(out / f"{name}.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow([plain(h) for h in header])
        w.writerows([[plain(c) for c in r] for r in rows])


# ------------------------------------------------------------------------------
# Tablas
# ------------------------------------------------------------------------------

def tables_eda(eda: dict, out: Path) -> None:
    names = {"train": "Entrenamiento", "val": "Validación", "test": "Prueba"}
    rows = []
    for s in ("train", "val", "test"):
        d = eda["longitud_tokens"][s]
        rows.append([names[s], num(eda["documentos"][s]), f"{d['media']:.0f}", f"{d['mediana']:.0f}",
                     f"{d['p75']:.0f}", f"{d['p90']:.0f}", num(d["maximo"]), str(eda["vacios"][s])])
    write_table(out, "eda_longitudes", ["Partición", "Documentos", "Media", "Mediana", "p75", "p90", "Máximo", "Vacíos"],
                rows, "lrrrrrrr")
    rows = [[f"L = {L}", f"{v:.1f}"] for L, v in eda["pct_completos_train"].items()]
    write_table(out, "eda_L", ["Longitud", "\\% de documentos sin truncar"], rows, "lr")
    rows = [[f"V = {int(V):,}".replace(",", "\\,"), f"{c * 100:.1f}"] for V, c in eda["cobertura_val"].items()]
    write_table(out, "eda_V", ["Vocabulario", "\\% de palabras de validación cubiertas"], rows, "lr")


def table_tuning(cell: str, tun: dict, out: Path) -> None:
    rows, bold = [], set()
    for k, step in enumerate(tun["pasos"], start=1):
        for t in step["pruebas"]:
            chosen = t["valor"] == step["elegido"]
            if chosen:
                bold.add(len(rows))
            rows.append([str(k), PARAM_LABELS[step["parametro"]], f"{t['valor']:g}", pct(t["val_f1_macro"]),
                         pct(t["val_accuracy"]), str(t["mejor_epoca"]), f"{t['s_por_epoca']:.1f}",
                         "$\\checkmark$" if chosen else ""])
    write_table(out, f"ajuste_{cell}", ["Paso", "Parámetro", "Valor", "F1 macro val. (\\%)", "Exactitud val. (\\%)",
                                        "Mejor época", "s/época", "Elegido"], rows, "clrrrrrc", bold)


def table_base_vs_tuned(tunings: dict, out: Path) -> None:
    rows = []
    for cell, tun in tunings.items():
        for label, cfg, val in (("base", tun["base_config"], tun["base_val"]),
                                ("ajustado", tun["config_final"], tun["final_val"])):
            rows.append([f"{NAMES[cell]} ({label})", str(cfg["seq_len"]), str(cfg["units"]), f"{cfg['dropout']:g}",
                         f"{cfg['lr']:g}", pct(val["f1_macro"]), pct(val["accuracy"])])
    write_table(out, "base_vs_ajustado", ["Modelo", "L", "Unidades", "Dropout", "lr", "F1 macro val. (\\%)",
                                          "Exactitud val. (\\%)"], rows, "lrrrrrr")


def table_final(root: Path, finals: dict, mlp_e2: dict | None, out: Path) -> list[dict]:
    rows_data = []
    for key, label in (("baseline_mlp", "MLP Baseline (E1)"),
                       ("tuned_mlp", "MLP Tuned (E1)")):
        r = load(root / "results" / f"{key}.json")
        if r:
            rows_data.append({"modelo": label, "params": None, "test": r, "best": None, "spe": None})
    if mlp_e2:
        rows_data.append({"modelo": MLP_E2, "params": mlp_e2.get("params"),
                          "test": mlp_e2, "best": mlp_e2.get("best_epoch"), "spe": mlp_e2.get("sec_per_epoch")})
    for cell in ("simple", "lstm"):
        f = finals.get(cell)
        if f:
            rows_data.append({"modelo": f"{NAMES[cell]} (ajustada)", "params": f["params"]["total"], "test": f["test"],
                              "best": f["best_epoch"], "spe": f["sec_per_epoch"]})
    rows = [[d["modelo"], num(d["params"]), pct(d["test"]["accuracy"]), pct(d["test"]["f1_macro"]),
             pct(d["test"]["precision_macro"]), pct(d["test"]["recall_macro"]),
             "—" if d["best"] is None else str(d["best"]), "—" if d["spe"] is None else f"{d['spe']:.1f}"]
            for d in rows_data]
    # Métricas en %: el pie de la tabla lo indica (encabezados cortos para que la tabla quepa sin reducirla)
    write_table(out, "final", ["Modelo", "Parámetros", "Exactitud", "F1 macro", "Precisión",
                               "Exhaustividad", "Mejor época", "s/época"], rows, "lrrrrrrr")
    return rows_data


def table_seeds(root: Path, out: Path) -> dict:
    res = {}
    rows = []
    for key, label in (("mlp", "MLP Tuned (re-entrenado)"), ("simple", "SimpleRNN (ajustada)"),
                       ("lstm", "LSTM (ajustada)")):
        s = load(root / "results" / "e2" / "final" / f"{key}_semillas.json")
        if s:
            res[key] = s
            seeds = ", ".join(str(r["seed"]) for r in s["por_semilla"])
            rows.append([label, f"{s['accuracy']['media'] * 100:.2f} $\\pm$ {s['accuracy']['desv'] * 100:.2f}",
                         f"{s['f1_macro']['media'] * 100:.2f} $\\pm$ {s['f1_macro']['desv'] * 100:.2f}", seeds])
    if rows:
        write_table(out, "semillas", ["Modelo", "Exactitud (\\%)", "F1 macro (\\%)", "Semillas"], rows, "lrrl")
    return res


def table_por_semilla(root: Path, out: Path, finals: dict) -> dict:
    """F1 macro de cada semilla, en validación y en prueba, con la configuración final.

    El modelo final de cada RNN es la semilla con mejor F1 macro de VALIDACIÓN (la prueba no
    interviene en la elección); su fila va en negrita. Se reportan también las demás semillas.
    """
    e2 = root / "results" / "e2"
    log_path = e2 / "bitacora.csv"
    if len(finals) < 2 or not log_path.exists():
        return {}
    with open(log_path, encoding="utf-8") as f:
        log = {r["run_id"]: r for r in csv.DictReader(f)}
    per = {}
    for cell in ("simple", "lstm"):
        s = load(e2 / "final" / f"{cell}_semillas.json")
        if not s:
            return {}
        base = finals[cell]["run_id"].rsplit("_s", 1)[0]
        per[cell] = {}
        for r in s["por_semilla"]:
            run = log.get(f"{base}_s{r['seed']}")
            val = float(run["val_f1_macro"]) if run and run.get("val_f1_macro") else None
            per[cell][r["seed"]] = (val, r["f1_macro"])
    mlp = load(e2 / "final" / "mlp_semillas.json")
    mlp_test = {r["seed"]: r["f1_macro"] for r in mlp["por_semilla"]} if mlp else {}
    seeds = list(per["simple"])
    chosen = {c: max(per[c], key=lambda k: per[c][k][0] if per[c][k][0] is not None else -1) for c in per}

    def pct(v):
        return "—" if v is None else f"{v * 100:.2f}"

    def mean_sd(vals):
        v = np.array([x for x in vals if x is not None]) * 100
        return f"{v.mean():.2f} $\\pm$ {v.std(ddof=1):.2f}" if len(v) > 1 else "—"

    rows, bold = [], set()
    for i, sd in enumerate(seeds):
        marks = [NAMES[c] for c in per if chosen[c] == sd]
        label = f"{sd} (final)" if marks else str(sd)
        if len(marks) == len(per):
            bold.add(i)
        rows.append([label, pct(per["simple"][sd][0]), pct(per["simple"][sd][1]),
                     pct(per["lstm"][sd][0]), pct(per["lstm"][sd][1]), pct(mlp_test.get(sd))])
    rows.append(["Media $\\pm$ d.e.",
                 mean_sd([per["simple"][k][0] for k in seeds]), mean_sd([per["simple"][k][1] for k in seeds]),
                 mean_sd([per["lstm"][k][0] for k in seeds]), mean_sd([per["lstm"][k][1] for k in seeds]),
                 mean_sd([mlp_test.get(k) for k in seeds])])
    write_table(out, "por_semilla", ["Semilla", "SimpleRNN val.", "SimpleRNN prueba", "LSTM val.",
                                     "LSTM prueba", "MLP prueba"], rows, "lrrrrr", bold_rows=bold)
    return {"semilla_final": chosen, "por_semilla": {c: {str(k): v for k, v in d.items()} for c, d in per.items()}}


# ------------------------------------------------------------------------------
# Figuras
# ------------------------------------------------------------------------------

def fig_validation(root: Path, finals: dict, figs: Path) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for cell, f in finals.items():
        h = load(root / "results" / "e2" / "runs" / f["run_id"] / "history.json")
        ep = np.arange(1, len(h["val_loss"]) + 1)
        for ax, key in zip(axes, ("val_loss", "val_accuracy")):
            ax.plot(ep, h[key], color=COLORS[cell], lw=2, marker="o", ms=4, label=NAMES[cell])
            b = f["best_epoch"]
            ax.plot([b], [h[key][b - 1]], "o", color=COLORS[cell], ms=9, mec="white", mew=1.5)
    axes[0].set_ylabel("Pérdida de validación")
    axes[1].set_ylabel("Exactitud de validación")
    for ax in axes:
        ax.set_xlabel("Época")
        style(ax)
        ax.legend(frameon=False)
    fig.suptitle("Validación por época (el punto grande marca la mejor época)", fontsize=11)
    fig.tight_layout()
    fig.savefig(figs / "comparacion_validacion.png", dpi=200)
    plt.close(fig)


def fig_tuning(tunings: dict, figs: Path) -> None:
    params = [s["parametro"] for s in next(iter(tunings.values()))["pasos"]]
    fig, axes = plt.subplots(1, len(params), figsize=(3.2 * len(params), 3.4), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, param in zip(axes, params):
        for cell, tun in tunings.items():
            step = next(s for s in tun["pasos"] if s["parametro"] == param)
            xs = [f"{t['valor']:g}" for t in step["pruebas"]]
            ys = [t["val_f1_macro"] * 100 for t in step["pruebas"]]
            ax.plot(xs, ys, color=COLORS[cell], lw=2, marker="o", ms=5, label=NAMES[cell])
            k = [t["valor"] for t in step["pruebas"]].index(step["elegido"])
            ax.plot([xs[k]], [ys[k]], "o", color=COLORS[cell], ms=10, mec="white", mew=1.5)
        ax.set_title(PARAM_LABELS[param], fontsize=10)
        style(ax)
    axes[0].set_ylabel("F1 macro de validación (%)")
    axes[0].legend(frameon=False, fontsize=8)
    fig.suptitle("Ajuste uno a la vez (el punto grande es el valor elegido)", fontsize=11)
    fig.tight_layout()
    fig.savefig(figs / "ajuste.png", dpi=200)
    plt.close(fig)


def fig_confusion(cm: np.ndarray, class_names: list[str], title: str, path: Path) -> None:
    norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(9.5, 8))
    im = ax.imshow(norm, cmap=BLUES, vmin=0, vmax=1)
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=90, fontsize=7)
    ax.set_yticklabels(class_names, fontsize=7)
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            if norm[i, j] >= 0.10:
                ax.text(j, i, f"{norm[i, j] * 100:.0f}", ha="center", va="center", fontsize=6,
                        color="white" if norm[i, j] > 0.5 else INK)
    ax.set_xlabel("Clase predicha")
    ax.set_ylabel("Clase real")
    ax.set_title(title, fontsize=11)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label("Proporción de la clase real")
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    plt.close(fig)


def length_analysis(root: Path, eda: dict, finals: dict, out: Path, figs: Path) -> dict | None:
    preds = root / "results" / "e2" / "preds"
    if not (preds / "test_lengths.npy").exists() or not (preds / "y_test.npy").exists():
        return None
    lengths = np.load(preds / "test_lengths.npy")
    y = np.load(preds / "y_test.npy")
    tr_len = eda["longitud_tokens"]["train"]
    cuts = [tr_len["p25"], tr_len["mediana"], tr_len["p75"]]
    edges = [-1] + cuts + [np.inf]
    labels = [f"$\\le$ {cuts[0]:.0f}", f"{cuts[0]:.0f}–{cuts[1]:.0f}", f"{cuts[1]:.0f}–{cuts[2]:.0f}",
              f"$>$ {cuts[2]:.0f}"]
    models = {}
    for key in ("mlp", "simple", "lstm"):
        f = preds / f"{key}_test_probs.npy"
        if f.exists():
            models[key] = np.load(f).argmax(axis=1)
    bins = np.digitize(lengths, edges[1:-1], right=True)
    # Una sola columna de truncados si las dos RNN usan la misma L
    Ls = {cell: finals[cell]["config"]["seq_len"] if cell in finals else None for cell in ("simple", "lstm")}
    trunc_L = [(None, Ls["simple"])] if Ls["simple"] == Ls["lstm"] else list(Ls.items())
    rows, acc = [], {k: [] for k in models}
    for b, lab in enumerate(labels):
        m = bins == b
        row = [lab + " tokens", str(int(m.sum()))]
        for cell, L in trunc_L:
            row.append("—" if L is None else f"{np.mean(lengths[m] > L) * 100:.0f}")
        for k, pred in models.items():
            a = float(np.mean(pred[m] == y[m])) if m.any() else float("nan")
            acc[k].append(a)
            row.append(pct(a, 1))
        rows.append(row)
    # Exactitud en %: el pie de la tabla lo indica
    header = ["Longitud", "Documentos"] + \
             [f"Truncados, L = {L} (\\%)" if cell is None else f"Truncados {NAMES[cell]} (\\%)"
              for cell, L in trunc_L] + [NAMES[k] for k in models]
    write_table(out, "longitud", header, rows, "l" + "r" * (len(header) - 1))

    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(len(labels))
    w = 0.8 / max(len(models), 1)
    for i, (k, vals) in enumerate(acc.items()):
        bars = ax.bar(x + (i - (len(models) - 1) / 2) * w, np.array(vals) * 100, w * 0.92,
                      color=COLORS[k], label=NAMES[k])
        for rect, v in zip(bars, vals):
            ax.annotate(f"{v * 100:.0f}", (rect.get_x() + rect.get_width() / 2, rect.get_height()),
                        xytext=(0, 2), textcoords="offset points", ha="center", fontsize=7, color=MUTED)
    ax.set_xticks(x)
    ax.set_xticklabels([l.replace("$\\le$", "≤").replace("$>$", ">") + " tokens" for l in labels])
    ax.set_ylabel("Exactitud en test (%)")
    ax.set_title("Exactitud según la longitud del documento (cuartiles de train)", fontsize=11)
    style(ax)
    ax.legend(frameon=False, ncol=len(models))
    fig.tight_layout()
    fig.savefig(figs / "longitud.png", dpi=200)
    plt.close(fig)
    return {"cortes": cuts, "exactitud": acc}


def fig_final_bars(rows_data: list[dict], figs: Path) -> None:
    keys = [(MLP_E2, "mlp"), ("SimpleRNN (ajustada)", "simple"),
            ("LSTM (ajustada)", "lstm")]
    present = [(d, k) for name, k in keys for d in rows_data if d["modelo"] == name]
    if not present:
        return
    fig, ax = plt.subplots(figsize=(7, 4))
    metrics_ = [("accuracy", "Exactitud"), ("f1_macro", "F1 macro")]
    x = np.arange(len(metrics_))
    w = 0.8 / len(present)
    for i, (d, k) in enumerate(present):
        vals = [d["test"][m] * 100 for m, _ in metrics_]
        bars = ax.bar(x + (i - (len(present) - 1) / 2) * w, vals, w * 0.92, color=COLORS[k], label=NAMES[k])
        for rect, v in zip(bars, vals):
            ax.annotate(f"{v:.1f}", (rect.get_x() + rect.get_width() / 2, rect.get_height()), xytext=(0, 2),
                        textcoords="offset points", ha="center", fontsize=8, color=MUTED)
    ax.set_xticks(x)
    ax.set_xticklabels([lab for _, lab in metrics_])
    ax.set_ylabel("Test (%)")
    ax.set_ylim(0, 100)
    ax.set_title("Resultado en test (semilla 42)", fontsize=11)
    style(ax)
    ax.legend(frameon=False, ncol=len(present))
    fig.tight_layout()
    fig.savefig(figs / "comparacion_final.png", dpi=200)
    plt.close(fig)


# ------------------------------------------------------------------------------
# Principal
# ------------------------------------------------------------------------------

def main(argv: list[str] | None = None) -> dict:
    p = argparse.ArgumentParser(description="Tablas y figuras de la Entrega 2")
    p.add_argument("--output-root", default=".")
    a = p.parse_args(argv)
    root = Path(a.output_root)
    e2 = root / "results" / "e2"
    out, figs = e2 / "tables", e2 / "figures"
    out.mkdir(parents=True, exist_ok=True)
    figs.mkdir(parents=True, exist_ok=True)

    eda = load(e2 / "eda.json")
    tunings = {c: t for c in ("simple", "lstm") if (t := load(e2 / f"ajuste_{c}.json"))}
    finals = {c: f for c in ("simple", "lstm") if (f := load(e2 / "final" / f"{c}.json"))}
    mlp_e2 = load(root / "results" / "tuned_mlp_e2.json")
    class_names = load(root / "models" / "e2" / "class_names.json")

    if eda:
        tables_eda(eda, out)
    for cell, tun in tunings.items():
        table_tuning(cell, tun, out)
    if tunings:
        table_base_vs_tuned(tunings, out)
        fig_tuning(tunings, figs)
    rows_data = table_final(root, finals, mlp_e2, out)
    fig_final_bars(rows_data, figs)
    seeds = table_seeds(root, out)
    seed_choice = table_por_semilla(root, out, finals)
    if len(finals) == 2:
        fig_validation(root, finals, figs)

    # Matriz de confusión del mejor modelo recurrente (por F1 macro en test) y F1 por clase
    summary = {"finales": {}, "semillas": seeds, "eleccion_semilla": seed_choice}
    if finals and class_names:
        best = max(finals, key=lambda c: finals[c]["test"]["f1_macro"])
        cm = np.load(e2 / "preds" / f"{best}_confusion.npy")
        fig_confusion(cm, class_names, f"Matriz de confusión normalizada: {NAMES[best]} (test)",
                      figs / "confusion_mejor.png")
        y = np.load(e2 / "preds" / "y_test.npy")
        f1s = {}
        for key in ("mlp", "simple", "lstm"):
            f = e2 / "preds" / f"{key}_test_probs.npy"
            if f.exists():
                f1s[key] = skm.f1_score(y, np.load(f).argmax(axis=1), average=None,
                                        labels=list(range(len(class_names))), zero_division=0)
        rows = [[c.replace("_", "\\_")] + [pct(float(f1s[k][i]), 1) for k in f1s] for i, c in enumerate(class_names)]
        write_table(out, "por_clase", ["Clase"] + [f"F1 {NAMES[k]} (\\%)" for k in f1s], rows, "l" + "r" * len(f1s))
        # Pares más confundidos del mejor modelo
        norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
        off = [(norm[i, j], class_names[i], class_names[j]) for i in range(len(cm)) for j in range(len(cm)) if i != j]
        summary["mejor_rnn"] = best
        summary["confusiones_principales"] = [{"real": r, "predicha": p_, "proporcion": float(v)}
                                              for v, r, p_ in sorted(off, reverse=True)[:6]]
        summary["f1_por_clase"] = {NAMES[k]: {c: float(v[i]) for i, c in enumerate(class_names)} for k, v in f1s.items()}

    if eda and finals:
        summary["longitud"] = length_analysis(root, eda, finals, out, figs)
    for cell, f in finals.items():
        summary["finales"][cell] = {k: f[k] for k in ("config", "params", "epochs_run", "best_epoch",
                                                      "sec_per_epoch", "train_time_sec", "val", "test")}
    if mlp_e2:
        summary["mlp_e2"] = {k: mlp_e2.get(k) for k in ("accuracy", "f1_macro", "precision_macro", "recall_macro",
                                                        "params", "best_epoch", "epochs_run", "sec_per_epoch")}
    if eda:
        summary["eda"] = {"longitud_train": eda["longitud_tokens"]["train"], "vacios": eda["vacios"],
                          "pct_completos_train": eda["pct_completos_train"], "cobertura_val": eda["cobertura_val"],
                          "razon_S_W": eda["razon_S_W"], "vocabulario_train_total": eda["vocabulario_train_total"]}
    with open(e2 / "resumen.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False, default=float)
    print(f"Tablas en {out} | Figuras en {figs} | Resumen en {e2 / 'resumen.json'}")
    return summary


if __name__ == "__main__":
    main()

In [ ]:
%%writefile demo_e2.py
"""
================================================================================
PROYECTO PRÁCTICO: REDES NEURONALES - UNIVERSIDAD DEL VALLE (2026-2)
ENTREGA 2: DEMO EJECUTABLE (MLP de la E1 vs. SimpleRNN vs. LSTM)
================================================================================
Clasifica el mismo texto con los tres modelos y muestra el top-3 de cada uno.
No necesita internet: usa los archivos de models/e2/ que deja run_e2.py.

Modos:
  python demo_e2.py --run-samples        casos de la E1 + par de orden + texto largo
  python demo_e2.py --text "..."         un texto
  python demo_e2.py                      modo interactivo
"""

from __future__ import annotations

import argparse
import contextlib
import io
import json
import os
import pickle
from pathlib import Path

import numpy as np

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import keras
import tensorflow as tf

import train_rnn as tr

MODELS = Path("models") / "e2"

SAMPLES = [
    ("NASA rover detects evidence of liquid water and atmosphere on Mars.", "Caso claro (E1)"),
    ("The engine needs a new oil filter and spark plugs for better mileage.", "Caso claro (E1)"),
    ("A debate on whether doctors should perform experimental medical treatments when religious beliefs oppose it.",
     "Caso mixto (E1): medicina vs. religión"),
    ("I installed a new operating system on my PC to play baseball and hockey video games.",
     "Caso mixto (E1): sistemas vs. deportes"),
    ("Congressional debate on government funding for public healthcare hospitals and health insurance.",
     "Caso mixto (E1): política vs. salud"),
    ("The doctor treated the patient without medicine.", "Par de orden (1/2)"),
    ("The patient treated the doctor without medicine.", "Par de orden (2/2)"),
]
FILLER = "I have been thinking about this for a while and I wanted to share a few thoughts with everyone here. "
KEY_SENTENCE = "Anyway, the goalie made forty saves and the hockey team won the playoff game in overtime."


class Classifier:
    def __init__(self, name: str, predict_fn, seq_len: int | None = None):
        self.name, self.predict_fn, self.seq_len = name, predict_fn, seq_len


def load_rnn(cell: str, class_names: list[str]) -> Classifier | None:
    cfg_file = MODELS / f"{cell}_final_config.json"
    if not cfg_file.exists():
        return None
    cfg = json.loads(cfg_file.read_text(encoding="utf-8"))
    vocab = (MODELS / f"vocab_V{cfg['vocab_size']}.txt").read_text(encoding="utf-8").split("\n")
    vec = keras.layers.TextVectorization(output_mode="int", vocabulary=vocab, output_sequence_length=cfg["seq_len"])
    model = tr.build_rnn(cell, len(vocab), cfg["embed_dim"], cfg["units"], cfg["dropout"],
                         num_classes=len(class_names))
    model.load_weights(MODELS / f"{cell}_final.weights.h5")
    empty_id = vocab.index(tr.EMPTY_TOKEN) if tr.EMPTY_TOKEN in vocab else 1

    def predict(texts):
        # Mismo preprocesamiento que en el entrenamiento (train_rnn.SequenceData.padded)
        X = vec(tf.constant(tr.fill_empty(texts))).numpy()
        X[(X != 0).sum(axis=1) == 0, 0] = empty_id
        return model.predict(X, verbose=0)

    return Classifier(tr.CELL_NAMES[cell], predict, cfg["seq_len"])


def load_mlp(class_names: list[str]) -> Classifier | None:
    cfg_file = MODELS / "mlp_config.json"
    if not cfg_file.exists():
        return None
    import train_mlp
    cfg = json.loads(cfg_file.read_text(encoding="utf-8"))
    with contextlib.redirect_stdout(io.StringIO()):
        model = train_mlp.build_mlp_model(cfg["vocab_size"], len(class_names), cfg["hidden_dim"], cfg["dropout"])
    model.load_weights(MODELS / "mlp_tuned.weights.h5")
    with open(MODELS / "mlp_vectorizer.pkl", "rb") as f:
        vectorizer = pickle.load(f)

    def predict(texts):
        return model.predict(vectorizer.transform(texts).toarray(), verbose=0)

    return Classifier("MLP (E1)", predict)


def show(text: str, label: str, classifiers: list[Classifier], class_names: list[str], n_tokens: int | None = None):
    print("\n" + "=" * 100)
    print(f"[{label}]")
    print(f"\"{text if len(text) < 160 else text[:110] + ' [...] ' + text[-60:]}\"")
    if n_tokens is not None:
        print(f"Palabras del texto: {n_tokens}")
    print("-" * 100)
    print(f"{'Modelo':<12}{'1.ª opción':<32}{'2.ª opción':<32}{'3.ª opción':<32}")
    for c in classifiers:
        probs = c.predict_fn([text])[0]
        top = np.argsort(probs)[::-1][:3]
        cells = [f"{class_names[i]} ({probs[i] * 100:.1f} %)" for i in top]
        note = f"   (solo lee las primeras {c.seq_len})" if (c.seq_len and n_tokens and n_tokens > c.seq_len) else ""
        print(f"{c.name:<12}" + "".join(f"{s:<32}" for s in cells) + note)


def main(argv: list[str] | None = None) -> None:
    p = argparse.ArgumentParser(description="Demo Entrega 2: MLP vs. SimpleRNN vs. LSTM (20 Newsgroups)")
    p.add_argument("--text", default=None, help="Texto en inglés a clasificar")
    p.add_argument("--run-samples", action="store_true", help="Casos preparados para la sustentación")
    a = p.parse_args(argv)

    class_names = json.loads((MODELS / "class_names.json").read_text(encoding="utf-8"))
    classifiers = [c for c in (load_mlp(class_names), load_rnn("simple", class_names), load_rnn("lstm", class_names))
                   if c is not None]
    print("Modelos cargados: " + ", ".join(c.name for c in classifiers))

    if a.run_samples:
        for text, label in SAMPLES:
            show(text, label, classifiers, class_names)
        max_L = max((c.seq_len or 0) for c in classifiers)
        long_text = FILLER * max(1, (max_L // 20) + 2) + KEY_SENTENCE
        show(long_text, "Texto largo: las palabras clave aparecen al final", classifiers, class_names,
             n_tokens=len(long_text.split()))
        return
    if a.text:
        show(a.text, "Texto", classifiers, class_names, n_tokens=len(a.text.split()))
        return

    print("\nEscribe un texto en inglés y presiona ENTER ('salir' para terminar).")
    while True:
        try:
            text = input("\nTexto > ").strip()
        except (KeyboardInterrupt, EOFError):
            break
        if not text:
            continue
        if text.lower() in {"salir", "exit", "quit"}:
            break
        show(text, "Texto", classifiers, class_names, n_tokens=len(text.split()))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile memoria_e2.py
"""
Entrega 2 - Prueba de memoria con distancia controlada.

Cada frase de prueba tiene un tema claro (su clase esperada) y va AL INICIO del texto;
después se agregan k palabras de un texto de relleno sin tema (se usan tres rellenos
distintos para que el resultado no dependa de uno solo). Como las RNN clasifican con su último
estado, la información del tema tiene que atravesar k pasos para llegar a la salida.
Se mide, para cada k, la probabilidad media de la clase esperada y el % de frases en que
esa clase sigue siendo la primera opción. El MLP (TF-IDF) no depende del orden: sirve de
referencia.

Usa los modelos guardados en models/e2 (no descarga datos). Salidas:
  results/e2/memoria.json
  results/e2/figures/memoria.png
  results/e2/tables/memoria.{csv,md,tex}

Uso:  python memoria_e2.py
"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np

import demo_e2 as demo
import train_rnn as tr

# Frases con un tema claro y la clase que les corresponde
FRASES = [
    ("NASA rover detects evidence of liquid water and atmosphere on Mars.", "sci.space"),
    ("The engine needs a new oil filter and spark plugs for better mileage.", "rec.autos"),
    (demo.KEY_SENTENCE.replace("Anyway, t", "T"), "rec.sport.hockey"),
    ("The doctor prescribed antibiotics because the patient had a bacterial infection.", "sci.med"),
    ("For sale: used laptop in good condition, asking 300 dollars plus shipping.", "misc.forsale"),
    ("The pitcher threw a perfect game and the batter hit a home run in the ninth inning.", "rec.sport.baseball"),
    ("Israeli and Palestinian leaders met to discuss the peace process in the region.", "talk.politics.mideast"),
    ("I need software to render 3D images and convert them to GIF and JPEG formats.", "comp.graphics"),
]
# Palabras de relleno agregadas DESPUÉS de la frase (todas caben en L = 400)
DISTANCIAS = [0, 10, 25, 50, 100, 150, 200, 300]
RELLENOS = [
    demo.FILLER.strip(),
    "Please let me know if you have any questions or comments about this and I will try to answer as soon as possible.",
    "Yesterday we went out for a long walk with some friends and then we talked for hours about many different things.",
]


def texto(frase: str, relleno: str, k: int) -> str:
    palabras = relleno.split()
    return " ".join([frase] + (palabras * (k // len(palabras) + 1))[:k])


def main() -> None:
    class_names = json.loads((demo.MODELS / "class_names.json").read_text(encoding="utf-8"))
    modelos = [m for m in (demo.load_mlp(class_names), demo.load_rnn("simple", class_names),
                           demo.load_rnn("lstm", class_names)) if m is not None]
    esperadas = [class_names.index(c) for _, c in FRASES]

    res = {"frases": [{"texto": f, "clase": c} for f, c in FRASES], "distancias": DISTANCIAS,
           "rellenos": RELLENOS, "casos_por_distancia": len(FRASES) * len(RELLENOS), "modelos": {}}
    y = np.array(esperadas * len(RELLENOS))
    for m in modelos:
        prob, acierto, por_relleno = [], [], []
        for k in DISTANCIAS:
            p = m.predict_fn([texto(f, r, k) for r in RELLENOS for f, _ in FRASES])
            ok = p.argmax(axis=1) == y
            prob.append(float(np.mean(p[np.arange(len(y)), y])))
            acierto.append(float(np.mean(ok)))
            por_relleno.append([float(np.mean(o)) for o in np.split(ok, len(RELLENOS))])
        res["modelos"][m.name] = {"prob_media": prob, "aciertos": acierto,
                                  "aciertos_por_relleno": por_relleno}

    out = tr.results_dir()
    (out / "figures").mkdir(parents=True, exist_ok=True)
    (out / "tables").mkdir(parents=True, exist_ok=True)
    (out / "memoria.json").write_text(json.dumps(res, indent=2, ensure_ascii=False), encoding="utf-8")

    # Tabla: % de frases con la clase esperada como primera opción
    nombres = list(res["modelos"])
    filas = [[str(k)] + [f"{res['modelos'][n]['aciertos'][i] * 100:.0f}" for n in nombres]
             for i, k in enumerate(DISTANCIAS)]
    cab = ["Palabras después de la frase"] + [f"{n.replace(' (E1)', '')}: aciertos (%)" for n in nombres]
    (out / "tables" / "memoria.csv").write_text(
        "\n".join(",".join(r) for r in [cab] + filas) + "\n", encoding="utf-8")
    (out / "tables" / "memoria.md").write_text(
        "| " + " | ".join(cab) + " |\n|" + "---|" * len(cab) + "\n"
        + "".join("| " + " | ".join(r) + " |\n" for r in filas), encoding="utf-8")
    tex = ["\\begin{tabular}{" + "r" * len(cab) + "}", "\\toprule",
           "\\textbf{Palabras después} & " + " & ".join(
               f"\\textbf{{{n.replace(' (E1)', '')}}}" for n in nombres) + " \\\\", "\\midrule"]
    tex += [" & ".join(r) + " \\\\" for r in filas] + ["\\bottomrule", "\\end{tabular}"]
    (out / "tables" / "memoria.tex").write_text("\n".join(tex) + "\n", encoding="utf-8")

    # Figura: probabilidad media de la clase esperada según la distancia
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    colores = {"MLP (E1)": tr.COLORS.get("mlp", "#1baf7a"), "SimpleRNN": tr.COLORS["simple"],
               "LSTM": tr.COLORS["lstm"]}
    fig, ax = plt.subplots(figsize=(8, 4), dpi=200)
    for n in nombres:
        y = np.array(res["modelos"][n]["prob_media"]) * 100
        ax.plot(DISTANCIAS, y, marker="o", markersize=5, linewidth=2,
                color=colores.get(n, "#555555"), label=n.replace(" (E1)", ""))
        ax.annotate(f"{y[-1]:.0f}", (DISTANCIAS[-1], y[-1]), textcoords="offset points",
                    xytext=(6, -3), fontsize=8, color="#444444")
    ax.axhline(100 / len(class_names), color="#888888", linewidth=1, linestyle=":",
               label=f"azar (1 de {len(class_names)})")
    ax.set_xlabel("Palabras de relleno después de la frase con el tema")
    ax.set_ylabel("Probabilidad media de la clase correcta (%)")
    ax.set_title(f"Prueba de memoria: {len(FRASES)} frases × {len(RELLENOS)} rellenos, la frase va al inicio")
    ax.set_ylim(0, 100)
    ax.set_xticks(DISTANCIAS)
    ax.grid(True, color="#e5e5e5", linewidth=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(out / "figures" / "memoria.png")
    plt.close(fig)

    print("Probabilidad media de la clase correcta (%) y aciertos (%) según la distancia")
    print("k".rjust(5) + "".join(f"{n:>24s}" for n in nombres))
    for i, k in enumerate(DISTANCIAS):
        print(f"{k:5d}" + "".join(
            f"{res['modelos'][n]['prob_media'][i] * 100:14.1f} / {res['modelos'][n]['aciertos'][i] * 100:5.0f}"
            for n in nombres))
    print(f"\nGuardado en {out / 'memoria.json'}, {out / 'figures' / 'memoria.png'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile train_mlp.py
"""
================================================================================
PROYECTO PRÁCTICO: REDES NEURONALES - UNIVERSIDAD DEL VALLE (2026-2)
ENTREGA 1: CLASIFICACIÓN DE TEXTO CON PERCEPTRÓN MULTICAPA (MLP)
DATASET 3: 20 NEWSGROUPS (20 CATEGORÍAS) | FRAMEWORK: KERAS
================================================================================
Correcciones hechas en la Entrega 2 (no cambian los resultados con semilla 42):
  - Las carpetas models/ y results/ se crean al inicio (antes fallaba en un clon limpio).
  - Nuevo argumento --seed (semilla de inicialización); la partición sigue fija en 42.
  - Las curvas numeran las épocas desde 1 y el JSON guarda el historial por época.
  - Nuevo argumento --probs-out para guardar las probabilidades de test (análisis E2).
"""

from __future__ import annotations

import argparse
import json
import os
import pickle
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import sklearn.metrics as metrics
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import keras
from keras import callbacks, layers, models, regularizers


# ==============================================================================
# SECCIÓN 1: CONFIGURACIÓN Y REPRODUCIBILIDAD
# ==============================================================================

def set_seed(seed: int = 42) -> None:
    """Fija semillas aleatorias para garantizar reproducibilidad."""
    random.seed(seed)
    np.random.seed(seed)
    keras.utils.set_random_seed(seed)


# ==============================================================================
# SECCIÓN 2: CARGA Y PREPROCESAMIENTO DE TEXTO (TF-IDF)
# ==============================================================================

def load_and_preprocess_data(
    vocab_size: int = 5000,
    val_split: float = 0.15,
    seed: int = 42,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, list[str], TfidfVectorizer]:
    """
    Carga el dataset 20 Newsgroups sin encabezados/firmas para evitar fugas de datos
    y transforma el texto crudo en vectores numéricos mediante TF-IDF.
    """
    print("\n--- SECCIÓN 2: CARGA Y VECTORIZACIÓN TF-IDF ---")
    
    # Remoción de metadatos (headers, footers, quotes) para evitar atajos espurios
    raw_train = fetch_20newsgroups(subset="train", remove=("headers", "footers", "quotes"), random_state=seed)
    raw_test = fetch_20newsgroups(subset="test", remove=("headers", "footers", "quotes"), random_state=seed)
    class_names = list(raw_train.target_names)

    # Partición estratificada: 85% entrenamiento y 15% validación
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        raw_train.data, raw_train.target, test_size=val_split, random_state=seed, stratify=raw_train.target
    )

    # Vectorizador TF-IDF con filtrado de stop-words y frecuencia de términos sublineal
    vectorizer = TfidfVectorizer(
        max_features=vocab_size,
        stop_words="english",
        sublinear_tf=True,
        min_df=2,
        dtype=np.float32,
    )

    X_train = vectorizer.fit_transform(train_texts).toarray()
    X_val = vectorizer.transform(val_texts).toarray()
    X_test = vectorizer.transform(raw_test.data).toarray()

    y_train = np.array(train_labels, dtype=np.int32)
    y_val = np.array(val_labels, dtype=np.int32)
    y_test = np.array(raw_test.target, dtype=np.int32)

    print(f"Vocabulario: {vocab_size} términos | Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
    return X_train, y_train, X_val, y_val, X_test, y_test, class_names, vectorizer


# ==============================================================================
# SECCIÓN 3: ARQUITECTURA DEL PERCEPTRÓN MULTICAPA (MLP)
# ==============================================================================

def build_mlp_model(
    input_dim: int,
    num_classes: int = 20,
    hidden_dim: int = 256,
    dropout_rate: float = 0.4,
    l2_reg: float = 1e-4,
    learning_rate: float = 1e-3,
) -> keras.Model:
    """
    Construye un MLP con 2 capas ocultas densas, activaciones ReLU, regularización L2,
    Dropout para controlar sobreajuste y capa de salida Softmax.
    """
    print("\n--- SECCIÓN 3: CONSTRUCCIÓN DEL MODELO MLP ---")

    reg = regularizers.l2(l2_reg) if l2_reg > 0 else None

    model = models.Sequential([
        layers.Input(shape=(input_dim,), name="tfidf_input"),
        
        # Capa Oculta 1: Extrae patrones de co-ocurrencia léxica
        layers.Dense(hidden_dim, kernel_regularizer=reg, activation="relu", name="dense_1"),
        layers.Dropout(dropout_rate, name="dropout_1"),

        # Capa Oculta 2: Comprime representaciones abstractas intermedias
        layers.Dense(hidden_dim // 2, kernel_regularizer=reg, activation="relu", name="dense_2"),
        layers.Dropout(dropout_rate, name="dropout_2"),

        # Capa de Salida: Distribución de probabilidad para las 20 clases
        layers.Dense(num_classes, activation="softmax", name="output_probabilities"),
    ], name="MLP_20Newsgroups")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    model.summary()
    return model


# ==============================================================================
# SECCIÓN 4: ENTRENAMIENTO Y CURVAS DE APRENDIZAJE
# ==============================================================================

def train_and_plot(
    model: keras.Model,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    epochs: int = 20,
    batch_size: int = 64,
    output_prefix: str = "baseline_mlp",
) -> tuple[keras.callbacks.History, float, str]:
    """Entrena la red con EarlyStopping y guarda únicamente las curvas de pérdida y exactitud."""
    print("\n--- SECCIÓN 4: ENTRENAMIENTO DEL MODELO ---")
    os.makedirs("models", exist_ok=True)
    os.makedirs("results", exist_ok=True)

    cb_list = [
        callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1),
        callbacks.ModelCheckpoint(f"models/{output_prefix}.keras", monitor="val_loss", save_best_only=True),
    ]

    start_time = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=cb_list,
        verbose=1,
    )
    total_time = time.time() - start_time
    print(f"Tiempo de entrenamiento: {total_time:.1f}s")

    # Gráfica única de Curvas de Pérdida y Aprendizaje (Train vs. Val); épocas numeradas desde 1
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
    ep = range(1, len(history.history["loss"]) + 1)

    ax1.plot(ep, history.history["loss"], label="Train Loss", color="#1f77b4", lw=2)
    ax1.plot(ep, history.history["val_loss"], label="Val Loss", color="#ff7f0e", lw=2)
    ax1.set_title("Curva de Pérdida (Cross-Entropy)")
    ax1.set_xlabel("Época")
    ax1.set_ylabel("Loss")
    ax1.grid(True, linestyle="--", alpha=0.5)
    ax1.legend()

    ax2.plot(ep, history.history["accuracy"], label="Train Accuracy", color="#2ca02c", lw=2)
    ax2.plot(ep, history.history["val_accuracy"], label="Val Accuracy", color="#d62728", lw=2)
    ax2.set_title("Curva de Aprendizaje (Accuracy)")
    ax2.set_xlabel("Época")
    ax2.set_ylabel("Exactitud")
    ax2.grid(True, linestyle="--", alpha=0.5)
    ax2.legend()

    plt.tight_layout()
    curves_path = f"results/learning_curves_{output_prefix}.png"
    plt.savefig(curves_path, dpi=200)
    plt.close()
    print(f"Curvas de pérdida y aprendizaje guardadas en: {curves_path}")

    return history, total_time, curves_path


# ==============================================================================
# SECCIÓN 5: EVALUACIÓN FORMAL DE MÉTRICAS
# ==============================================================================

def evaluate_model(
    model: keras.Model,
    X_test: np.ndarray,
    y_test: np.ndarray,
    class_names: list[str],
    probs_out: str | None = None,
) -> dict:
    """Evalúa el modelo sobre el conjunto de test independiente y extrae métricas cuantitativas."""
    print("\n--- SECCIÓN 5: EVALUACIÓN EN CONJUNTO DE PRUEBA ---")
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    probs = model.predict(X_test, batch_size=128, verbose=0)
    y_pred = np.argmax(probs, axis=1)
    if probs_out:
        Path(probs_out).parent.mkdir(parents=True, exist_ok=True)
        np.save(probs_out, probs.astype(np.float32))

    macro_f1 = metrics.f1_score(y_test, y_pred, average="macro", zero_division=0)
    weighted_f1 = metrics.f1_score(y_test, y_pred, average="weighted", zero_division=0)
    precision = metrics.precision_score(y_test, y_pred, average="macro", zero_division=0)
    recall = metrics.recall_score(y_test, y_pred, average="macro", zero_division=0)

    print(f"Exactitud (Accuracy):  {test_acc:.4f} ({test_acc * 100:.2f}%)")
    print(f"F1-Score (Macro):      {macro_f1:.4f}")
    print(f"F1-Score (Ponderado):  {weighted_f1:.4f}")
    print(f"Precisión (Macro):     {precision:.4f}")
    print(f"Sensibilidad (Recall): {recall:.4f}")

    return {
        "test_loss": float(test_loss),
        "accuracy": float(test_acc),
        "f1_macro": float(macro_f1),
        "f1_weighted": float(weighted_f1),
        "precision_macro": float(precision),
        "recall_macro": float(recall),
    }


# ==============================================================================
# SECCIÓN 6: DEMO DE INFERENCIA Y FUNCIÓN PRINCIPAL (CLI)
# ==============================================================================

def run_quick_demo(model: keras.Model, vectorizer: TfidfVectorizer, class_names: list[str]) -> None:
    """
    Demostración de inferencia con casos claros (alta certeza) y casos mixtos/ambiguos
    (certeza promedio y distribución competitiva de probabilidades).
    """
    print("\n--- SECCIÓN 6: DEMO DE INFERENCIA (CASOS CLAROS Y MIXTOS) ---")
    samples = [
        # 1 y 2: Casos claros (alta confianza)
        ("NASA rover detects evidence of liquid water and atmosphere on Mars.", "Claro"),
        ("The engine needs a new oil filter and spark plugs for better mileage.", "Claro"),
        
        # 3, 4 y 5: Casos mixtos / frontera (confianza promedio con clases competidoras)
        ("A debate on whether doctors should perform experimental medical treatments when religious beliefs oppose it.", "Mixto (Medicina vs. Religión/Ética)"),
        ("I installed a new operating system on my PC to play baseball and hockey video games.", "Mixto (Sistemas vs. Deportes)"),
        ("Congressional debate on government funding for public healthcare hospitals and health insurance.", "Mixto (Política vs. Salud Pública)"),
    ]

    preds = model.predict(vectorizer.transform([s[0] for s in samples]).toarray(), verbose=0)
    for (text, tipo), prob in zip(samples, preds):
        top_idx = int(np.argmax(prob))
        top_indices = np.argsort(prob)[-3:][::-1]
        top3_str = ", ".join([f"{class_names[i]} ({prob[i]*100:.1f}%)" for i in top_indices])
        print(f"\n[{tipo}] \"{text}\"")
        print(f"  -> Predicción principal: {class_names[top_idx]} ({prob[top_idx]*100:.1f}%)")
        print(f"  -> Top 3 alternativas:   {top3_str}")


def main(argv: list[str] | None = None) -> None:
    parser = argparse.ArgumentParser(description="Clasificador MLP en Keras para 20 Newsgroups (Entrega 1)")
    parser.add_argument("--vocab-size", type=int, default=5000, help="Tamaño del vocabulario TF-IDF")
    parser.add_argument("--hidden-dim", type=int, default=256, help="Neuronas en primera capa oculta")
    parser.add_argument("--dropout", type=float, default=0.4, help="Tasa de Dropout (regularización)")
    parser.add_argument("--l2-reg", type=float, default=1e-4, help="Penalización L2 (Weight Decay)")
    parser.add_argument("--lr", type=float, default=1e-3, help="Learning Rate")
    parser.add_argument("--epochs", type=int, default=20, help="Número de épocas")
    parser.add_argument("--batch-size", type=int, default=64, help="Tamaño de lote")
    parser.add_argument("--output-name", type=str, default="baseline_mlp", help="Nombre del modelo y resultados")
    parser.add_argument("--quick-test", action="store_true", help="Prueba rápida de 2 épocas")
    parser.add_argument("--seed", type=int, default=42, help="Semilla de inicialización (la partición queda fija en 42)")
    parser.add_argument("--probs-out", type=str, default=None, help="Ruta .npy para guardar las probabilidades de test")
    parser.add_argument("--no-demo", action="store_true", help="No ejecuta la demo al final")
    args = parser.parse_args(argv)

    if args.quick_test:
        args.epochs = 2
        args.vocab_size = 1500

    # Corrección E2: las carpetas se crean antes de guardar el vectorizador
    os.makedirs("models", exist_ok=True)
    os.makedirs("results", exist_ok=True)

    set_seed(args.seed)

    # 1. Cargar y procesar datos (partición fija con semilla 42, igual que en la E1)
    X_train, y_train, X_val, y_val, X_test, y_test, class_names, vectorizer = load_and_preprocess_data(
        vocab_size=args.vocab_size, seed=42
    )

    # Serializar vectorizador para inferencia futura
    with open(f"models/vectorizer_{args.output_name}.pkl", "wb") as f:
        pickle.dump(vectorizer, f)

    # 2. Construir arquitectura MLP
    model = build_mlp_model(
        input_dim=X_train.shape[1],   # = vocab_size cuando el corpus tiene al menos esa cantidad de términos
        num_classes=len(class_names),
        hidden_dim=args.hidden_dim,
        dropout_rate=args.dropout,
        l2_reg=args.l2_reg,
        learning_rate=args.lr,
    )

    # 3. Entrenar y graficar curvas
    history, train_time, curves_path = train_and_plot(
        model, X_train, y_train, X_val, y_val,
        epochs=args.epochs, batch_size=args.batch_size, output_prefix=args.output_name
    )

    # 4. Evaluación en Test
    results = evaluate_model(model, X_test, y_test, class_names, probs_out=args.probs_out)
    hist = {k: [float(v) for v in vals] for k, vals in history.history.items()}
    results.update({
        "model_name": args.output_name,
        "vocab_size": args.vocab_size,
        "hidden_dim": args.hidden_dim,
        "dropout": args.dropout,
        "l2_reg": args.l2_reg,
        "lr": args.lr,
        "epochs": args.epochs,
        "batch_size": args.batch_size,
        "seed": args.seed,
        "params": int(model.count_params()),
        "epochs_run": len(hist["loss"]),
        "best_epoch": int(np.argmin(hist["val_loss"]) + 1),
        "training_time_sec": train_time,
        "sec_per_epoch": train_time / len(hist["loss"]),
        "curves_path": curves_path,
        "history": hist,
    })

    # Guardar métricas en JSON
    with open(f"results/{args.output_name}.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)
    print(f"\nMétricas consolidadas guardadas en: results/{args.output_name}.json")

    # 5. Demostración en vivo
    if not args.no_demo:
        run_quick_demo(model, vectorizer, class_names)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile compare_results.py
"""
================================================================================
PROYECTO PRÁCTICO: REDES NEURONALES - UNIVERSIDAD DEL VALLE (2026-2)
ENTREGA 1: COMPARACIÓN DE RESULTADOS ENTRE VARIANTES DE MLP
================================================================================
Compara experimentos (por ejemplo, Baseline MLP vs. Tuned MLP con BatchNormalization/Dropout)
generando un reporte en Markdown y una gráfica comparativa de métricas.
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def load_json_metrics(path: Path) -> dict | None:
    if not path.exists():
        print(f"Advertencia: Archivo no encontrado en {path.resolve()}")
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def main() -> None:
    parser = argparse.ArgumentParser(description="Comparador de Métricas de Experimentos MLP (Entrega 1)")
    parser.add_argument("--model-a", type=str, default="results/baseline_mlp.json", help="Ruta al JSON del modelo base")
    parser.add_argument("--model-b", type=str, default="results/tuned_mlp.json", help="Ruta al JSON del modelo comparado")
    parser.add_argument("--output-report", type=str, default="results/comparison_report.md", help="Ruta del reporte Markdown")
    parser.add_argument("--output-img", type=str, default="results/metrics_comparison.png", help="Ruta del gráfico comparativo")
    args = parser.parse_args()

    m_a = load_json_metrics(Path(args.model_a))
    m_b = load_json_metrics(Path(args.model_b))

    if m_a is None and m_b is None:
        print("Error: No se encontró ninguno de los archivos JSON de métricas especificados.")
        return

    name_a = m_a.get("model_name", "Modelo A") if m_a else "Modelo A"
    name_b = m_b.get("model_name", "Modelo B") if m_b else "Modelo B"

    print(f"\nComparando experimentos: '{name_a}' vs. '{name_b}'...\n")

    def get_val(data, keys, fmt=".4f"):
        if data is None:
            return "N/A"
        val = data
        for k in keys:
            if not isinstance(val, dict):
                val = None
                break
            val = val.get(k)
        if val is None and isinstance(data, dict):
            val = data.get(keys[-1])
        if val is None:
            return "N/A"
        if isinstance(val, float):
            return f"{val:{fmt}}"
        return str(val)

    # Generación de la tabla Markdown
    table_lines = [
        "# Comparación de Desempeño: Experimentos MLP (Dataset 3: 20 Newsgroups)",
        "",
        "Este reporte técnico compara cuantitativamente el desempeño de diferentes configuraciones de red neuronal MLP.",
        "",
        f"| Métrica / Parámetro | {name_a} | {name_b} |",
        "| :--- | :---: | :---: |",
        f"| **Tamaño Vocabulario TF-IDF** | {get_val(m_a, ['vocab_size'])} | {get_val(m_b, ['vocab_size'])} |",
        f"| **Neuronas (Capa 1)** | {get_val(m_a, ['hidden_dim'])} | {get_val(m_b, ['hidden_dim'])} |",
        # Hiperparámetros con formato "g": con ".4f", un L2 de 5e-05 se mostraba como 0.0001
        f"| **Tasa Dropout** | {get_val(m_a, ['dropout'], 'g')} | {get_val(m_b, ['dropout'], 'g')} |",
        f"| **Regularización L2** | {get_val(m_a, ['l2_reg'], 'g')} | {get_val(m_b, ['l2_reg'], 'g')} |",
        f"| **Learning Rate** | {get_val(m_a, ['lr'], 'g')} | {get_val(m_b, ['lr'], 'g')} |",
        f"| **Batch Size** | {get_val(m_a, ['batch_size'])} | {get_val(m_b, ['batch_size'])} |",
        "| --- | --- | --- |",
        f"| **Exactitud (Test Accuracy)** | {get_val(m_a, ['accuracy'])} | {get_val(m_b, ['accuracy'])} |",
        f"| **F1-Score (Macro)** | {get_val(m_a, ['f1_macro'])} | {get_val(m_b, ['f1_macro'])} |",
        f"| **F1-Score (Weighted)** | {get_val(m_a, ['f1_weighted'])} | {get_val(m_b, ['f1_weighted'])} |",
        f"| **Precisión (Macro)** | {get_val(m_a, ['precision_macro'])} | {get_val(m_b, ['precision_macro'])} |",
        f"| **Sensibilidad (Macro Recall)** | {get_val(m_a, ['recall_macro'])} | {get_val(m_b, ['recall_macro'])} |",
        f"| **Tiempo de Entrenamiento** | {get_val(m_a, ['training_time_sec'], '.2f')}s | {get_val(m_b, ['training_time_sec'], '.2f')}s |",
        f"| **Épocas Máximas** | {get_val(m_a, ['epochs'])} | {get_val(m_b, ['epochs'])} |",
    ]

    report_content = "\n".join(table_lines)
    print(report_content)

    rep_path = Path(args.output_report)
    rep_path.parent.mkdir(parents=True, exist_ok=True)
    with open(rep_path, "w", encoding="utf-8") as f:
        f.write(report_content + "\n")
    print(f"\nReporte Markdown guardado en: {rep_path.resolve()}")

    # Generación de gráfica comparativa
    if m_a and m_b:
        metrics_names = ["Accuracy", "F1 (Macro)", "Precision (Macro)", "Recall (Macro)"]
        keys = ["accuracy", "f1_macro", "precision_macro", "recall_macro"]

        def get_metric(data, k):
            if not data:
                return 0.0
            if "test_metrics" in data and isinstance(data["test_metrics"], dict):
                return float(data["test_metrics"].get(k, data.get(k, 0.0)))
            return float(data.get(k, 0.0))

        vals_a = [get_metric(m_a, k) for k in keys]
        vals_b = [get_metric(m_b, k) for k in keys]

        x = np.arange(len(metrics_names))
        width = 0.35

        fig, ax = plt.subplots(figsize=(8, 5))
        rects1 = ax.bar(x - width/2, vals_a, width, label=name_a, color="#3498db")
        rects2 = ax.bar(x + width/2, vals_b, width, label=name_b, color="#2ecc71")

        ax.set_ylabel("Puntuación (0 a 1)")
        ax.set_title("Comparativa de Desempeño: Experimentos MLP")
        ax.set_xticks(x)
        ax.set_xticklabels(metrics_names)
        ax.set_ylim(0, 1.1)
        ax.legend()
        ax.grid(axis="y", linestyle="--", alpha=0.5)

        for rect in rects1 + rects2:
            h = rect.get_height()
            ax.annotate(f"{h:.3f}",
                        xy=(rect.get_x() + rect.get_width() / 2, h),
                        xytext=(0, 3), textcoords="offset points",
                        ha="center", va="bottom", fontsize=8)

        plt.tight_layout()
        img_path = Path(args.output_img)
        plt.savefig(img_path, dpi=200)
        plt.close()
        print(f"Gráfica comparativa guardada en: {img_path.resolve()}\n")


if __name__ == "__main__":
    main()

Resultados originales de la Entrega 1 (para la tabla comparativa):

In [ ]:
%%writefile results/baseline_mlp.json
{
  "test_loss": 1.3587206602096558,
  "accuracy": 0.6445831060409546,
  "f1_macro": 0.6295924171714761,
  "f1_weighted": 0.6424205161421137,
  "precision_macro": 0.6481478264111459,
  "recall_macro": 0.630269721412649,
  "model_name": "baseline_mlp",
  "vocab_size": 5000,
  "hidden_dim": 256,
  "dropout": 0.4,
  "l2_reg": 0.0001,
  "lr": 0.001,
  "epochs": 20,
  "batch_size": 64,
  "training_time_sec": 35.68626379966736,
  "curves_path": "results/learning_curves_baseline_mlp.png"
}

In [ ]:
%%writefile results/tuned_mlp.json
{
  "test_loss": 1.2086470127105713,
  "accuracy": 0.6802973747253418,
  "f1_macro": 0.6689671629319205,
  "f1_weighted": 0.680634272550702,
  "precision_macro": 0.6829770495182909,
  "recall_macro": 0.6679155138506698,
  "model_name": "tuned_mlp",
  "vocab_size": 10000,
  "hidden_dim": 512,
  "dropout": 0.3,
  "l2_reg": 5e-05,
  "lr": 0.0005,
  "epochs": 25,
  "batch_size": 64,
  "training_time_sec": 133.4211642742157,
  "curves_path": "results/learning_curves_tuned_mlp.png"
}

## 3. Prueba rápida (2–4 minutos)
Corre todo el flujo con pocas épocas en la carpeta `prueba_rapida/` (no se mezcla con los resultados reales).
Si esta celda termina sin errores, la ejecución completa también debería funcionar.

In [ ]:
!python run_e2.py --quick-test

## 4. Ejecución completa (la parte larga)

In [ ]:
!python run_e2.py

## 5. Tablas, figuras, prueba de memoria y comparación de la E1

In [ ]:
!python analyze_e2.py
!python memoria_e2.py
!python compare_results.py

In [ ]:
from IPython.display import Image, display
for f in ["eda_longitudes_vocabulario", "ajuste", "comparacion_validacion", "comparacion_final", "memoria",
          "longitud", "confusion_mejor"]:
    path = f"results/e2/figures/{f}.png"
    if os.path.exists(path):
        print(path)
        display(Image(path, width=900))
print(open("results/e2/tables/final.md", encoding="utf-8").read())
print(open("results/e2/tables/semillas.md", encoding="utf-8").read())

## 6. Demo (la misma que se mostrará en la presentación)

In [ ]:
!python demo_e2.py --run-samples

## 7. Descargar resultados
Descarga `resultados_e2.zip`. **Ponlo en la carpeta `proyecto2`**.

In [ ]:
!rm -f resultados_e2.zip
!zip -r -q resultados_e2.zip results models/e2 *.py
!ls -lh resultados_e2.zip
from google.colab import files
files.download("resultados_e2.zip")